In [1]:
%%writefile HellaSwag.py
import os
import json
import requests
import tiktoken
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.nn import functional as F
from transformers import GPT2LMHeadModel


DATA_CACHE_DIR = os.path.join(os.path.dirname(__file__), "HellaSwag")


def download_file(url, fname, chunk_size=1024):
    resp  =  requests.get(url, stream=True)
    total = int(resp.headers.get('content-length', 0)) #  resp.headers -> dictionary of HTTP response headers sent by the server / .get("content-length", 0) -> gets the file size in bytes from the header, returns 0 if header is missing (some servers don't send it)
    with open(fname, 'wb') as f, tqdm(desc=fname, total=total, unit='iB', unit_scale=True, unit_divisor=1024,) as bar:
        for data in resp.iter_content(chunk_size=chunk_size):  #  resp.iter_content(chunk_size=1024) -> yields raw bytes in pieces of 1024 bytes at a time
            size = f.write(data)  # > writes those raw bytes to disk & returns the number of bytes actually written
            bar.update(size) # update how many bytes were just written


hellaswags = {
    "train": "https://raw.githubusercontent.com/rowanz/hellaswag/master/data/hellaswag_train.jsonl",
    "val": "https://raw.githubusercontent.com/rowanz/hellaswag/master/data/hellaswag_val.jsonl",
    "test": "https://raw.githubusercontent.com/rowanz/hellaswag/master/data/hellaswag_test.jsonl",
}


enc = tiktoken.get_encoding("gpt2")


def download(split):
    os.makedirs(DATA_CACHE_DIR, exist_ok=True)
    data_url = hellaswags[split]
    data_filename = os.path.join(DATA_CACHE_DIR, f"hellaswag_{split}.jsonl")
    if not os.path.exists(data_filename):
        print(f"Downloading {split} data from {data_url} to {data_filename}")
        download_file(data_url, data_filename)


def render_example(example):
    # example is a dict from the jsonl file, looks like:
    # {
    #   "ctx": "The woman picked up the ball and",
    #   "label": 2,           <- index of the correct ending (0,1,2, or 3)
    #   "endings": [          <- always exactly 4 possible completions
    #       "threw it away",
    #       "sat down quietly",
    #       "threw it to her friend",
    #       "read a book"
    #   ]
    # }

    ctx = example["ctx"]
    label = example["label"]
    endings = example["endings"]

    data = {
        'label': label,
        'ctx_tokens' : None,
        'endings_tokens' : [],
    }

    ctx_tokens = enc.encode(ctx)
    data["ctx_tokens"] = ctx_tokens

    tok_rows = []
    mask_rows = []
    # tok_rows  -> will hold 4 lists, each being [ctx_tokens + one_ending_tokens]
    # mask_rows -> will hold 4 lists, each being [0,0,0,...,1,1,1]
    #              zeros over the context, ones over the ending,  so we can later evaluate loss ONLY on the ending part



    for ending in endings:
        end_tokens = enc.encode(' ' + ending)
        # prepends a space before the ending before tokenizing
        # this is because GPT-2's tokenizer was trained on text where
        # words after a space get different token ids than words at the start
        # e.g. "friend" and " friend" are different tokens in GPT-2's vocab
        # so prepending the space gives the correct tokenization for a word
        # that would naturally follow the context

        tok_rows.append(ctx_tokens + end_tokens)
        mask_rows.append(len(ctx_tokens)*[0] + len(end_tokens)*[1])

        data['endings_tokens'].append(end_tokens)       # store each ending's tokens for debugging


    max_len = max(len(row) for row in tok_rows)     # needed because the 4 endings have different numbers of tokens, and we need all rows to be the same length to form a tensor

    tokens = torch.zeros((4, max_len), dtype=torch.long) # we have 4 choices in each question
    mask = torch.zeros((4, max_len), dtype=torch.long)

    for i, (tok_row, mask_row) in enumerate(zip(tok_rows, mask_rows)):
        tokens[i, :len(tok_row)] = torch.tensor(tok_row) # tokens[i, :len(tok_row)] -> selects row i, columns 0 to len(tok_row) |   = torch.tensor(tok_row) -> fills those positions with this row's token ids
        mask[i, :len(tok_row)] = torch.tensor(mask_row)      # columns beyond len(tok_row) stay as 0 (padding from torch.zeros above)


    return data, tokens, mask, label
    # data   -> debug dict
    # tokens -> (4, max_len) tensor of token ids, one row per candidate ending
    # mask   -> (4, max_len) tensor of 0s and 1s marking ending positions
    # label  -> integer 0-3, which row is the correct ending


def iterate_examples(split):
    download(split)
    with open(os.path.join(DATA_CACHE_DIR, f"hellaswag_{split}.jsonl"), "r") as f:
        for line in f:
            # iterates over the file one line at a time
            # each `line` is a string like '{"ctx": "...", "label": 2, "endings": [...]}\n'
            example = json.loads(line)
            yield example
            # yield makes this a GENERATOR function instead of a regular function
            # instead of returning all examples at once (which would load everything into RAM),
            # it pauses here and gives back one example at a time
            # the caller gets one example, processes it, then this resumes for the next one
            # memory efficient for large datasets


@torch.no_grad()
def evaluate(model_type, device):
    torch.set_float32_matmul_precision('high')     # tells PyTorch to use TF32 precision on Ampere GPUs (A100, RTX 3090, etc.) | TF32 is faster than full FP32 with negligible accuracy loss | 'high' enables TF32, 'highest' forces full FP32
    model = GPT2LMHeadModel.from_pretrained(model_type)
    model.to(device)
    # loads HuggingFace's pretrained GPT-2 and moves to GPU/CPU
    num_correct_norm = 0   # counts correct predictions using length-normalized loss
    num_correct = 0        # counts correct predictions using raw total loss
    num_total = 0          # counts total examples seen


    for example in iterate_examples("val"):
        # calls our generator above, gets one example dict at a time
        data, tokens, mask, label = render_example(example)
        tokens = tokens.to(device)
        mask = mask.to(device)
        logits = model(tokens).logits # shape: (4, max_len, vocab_size)

        shift_logits = (logits[:,:-1,:]).contiguous()    # .contiguous() -> ensures the tensor's memory layout is sequential after slicing some PyTorch operations require this
                                                         #  removing the last position means we drop the prediction AFTER the sequence ends
        shift_tokens = (tokens[..., 1:]).contiguous()    #  tokens[..., 1:] -> removes the FIRST token, keeps everything from position 1 onward ->  aligns with shift_logits so that position i's logits are compared against token i+1

        shift_logits = shift_logits.view(-1, shift_logits.size(-1))

        shift_tokens = shift_tokens.view(-1)

        loss = F.cross_entropy(shift_logits, shift_tokens, reduction='none')     # reduction='none' -> returns one loss value per position instead of averaging

        reshaped_loss = loss.view(tokens.size(0), -1)

        shift_mask = mask[...,1:].contiguous()

        masked_shift_loss = reshaped_loss * shift_mask

        sum_loss = torch.sum(masked_shift_loss, dim=-1)

        avg_loss = sum_loss / shift_mask.sum(-1)   # shift_mask.sum(dim=1) -> counts how many 1s are in each row,  i.e. how many ending tokens each candidate has, without this, longer endings would always have higher total loss

        preds = torch.argmin(sum_loss).item()

        norm_preds = torch.argmin(avg_loss).item()

        num_total += 1

        num_correct += int(preds == label)

        num_correct_norm += int(norm_preds == label)

        print(f"{num_total} acc_norm: {num_correct_norm}/{num_total}={num_correct_norm / num_total:.4f}")

        if num_total < 10:
            # only prints detailed debug info for the first 9 examples
            print(f"Context:\n {example['ctx']}")
            print(f"Endings:")
            # \n inside the string is a newline character
            for i, end in enumerate(example["endings"]):
                print(f"{i} (loss: {avg_loss[i].item():.4f}) {end}")
                # avg_loss[i] -> indexes into the 4-element tensor to get loss for ending i
                # .item() -> converts tensor element to plain Python float
            print(f"predicted: {norm_preds}, actual: {label}")



if __name__ == "__main__":
# this block ONLY runs if you execute this file directly: python hellaswag.py
# it does NOT run if another file imports this file as a module
# standard Python pattern to separate "runnable script" from "importable module"

    import argparse
    parser = argparse.ArgumentParser()
    # argparse lets you pass command line arguments when running the script
    # e.g. python hellaswag.py -m gpt2-medium -d cuda

    parser.add_argument("-m", "--model_type", type=str, default="gpt2", help="the model type to use")
    # -m is the short flag, --model_type is the long flag, both do the same thing
    # type=str -> convert the input to string
    # default="gpt2" -> if you don't pass -m, it uses "gpt2"
    # help="..." -> shown when you run python hellaswag.py --help

    parser.add_argument("-d", "--device", type=str, default="cuda", help="the device to use")

    args = parser.parse_args()
    # actually reads sys.argv (the command line you typed) and populates args
    # args.model_type -> whatever you passed with -m
    # args.device -> whatever you passed with -d

    evaluate(args.model_type, args.device)
    # calls the main function with the parsed arguments

Writing HellaSwag.py


In [2]:
%%writefile train_gpt2.py
import os
import math
import time
import inspect
import tiktoken
import numpy as np

from torch.distributed import init_process_group, destroy_process_group
from torch.nn.parallel import DistributedDataParallel as DDP
import torch.distributed as dist

from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F
from functools import partial
from HellaSwag import render_example, iterate_examples

###_________________________ DISTRIBUTED TRAINING _______________________________________

ddp = int(os.environ.get('RANK', -1)) != -1
if ddp:
    assert torch.cuda.is_available(), f'There is not cude for the device to run ddp'
    init_process_group(backend='nccl')
    ddp_rank = int(os.environ['RANK'])
    ddp_local_rank = int(os.environ['LOCAL_RANK'])
    ddp_world_size = int(os.environ['WORLD_SIZE'])
    device = f'cuda:{ddp_local_rank}'
    torch.cuda.set_device(device)
    master_process = ddp_rank == 0
else:
    ddp_rank = 0
    ddp_local_rank = 0
    ddp_world_size = 1
    master_process = True
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Using device: {device}')


###_________________________ HYPERPARAMETERS _______________________________________

@dataclass
class GPT_config:
    ### for model
    n_embd: int = 1024
    n_head: int = 8
    n_layer: int = 8
    vocab_size: int = 50304
    block_size: int = 512  # Just a number that is divisible by 2 more times that the standard 50257
    ### for data
    bs: int = 16                          # per-GPU micro batch
    tokenizer: str = 'gpt2'
    tot_bs_for_grad_accum: int = 131072       
    # for the training shards
    data_root: str = "/kaggle/input/datasets/seif222/gpt-train-kaggle-zero-to-hero"   # <- point at your actual shards
    ### for optim sched
    max_lr: float = 6e-4
    min_lr_ratio: float = 0.1
    warmup_steps: int = 10
    max_steps: int = 900
    ### for optim
    beta1: float = 0.9
    beta2: float = 0.95
    eps: float = 1e-8
    weight_decay: float = 0.1
    # if using the MuonAdamW
    red_dim: int = -1
    steps: int = 5
    Nesterov: bool = True
    ### for training & Validation
    training_steps: int = 1000                 # = max_steps
    val_after_step: int = 100
    val_loss_accum_steps: int = 5
    ### CheckPoint
    checkpoint_after_steps: int = 50
    ### device
    device: str = device
    ### Multi-Device Training
    process_rank: int = ddp_rank
    num_processes: int = ddp_world_size
    ### Options
    # validaiton
    validation: bool = True
    # sampling
    model_sampling: bool = True
    num_sequence: int = 3
    max_length: int = 40
    #compile
    use_compile: bool = False                # note: compile breaks HellaSwag/gen



##_____________________________________ OPTIMIZER __________________________________

# class MuonAdamW:
#     def __init__(self, params, lr, beta2, red_dim, steps=5 , beta1=0.9, Nesterov=True, eps=1e-7):
#         self.params = params
#         self.lr = lr
#         self.beta2 = beta2
#         self.red_dim = red_dim
#         self.steps = steps
#         self.beta1 = beta1
#         self.Nesterov = Nesterov
#         self.eps = eps
#         self.i = 0

#     def step(self, lr=None):
#         if lr is None: lr = self.lr
#         with torch.no_grad():
#             for g in self.params:
#                 use_muon = g.get('use_muon', None)
#                 for p in g['params']:
#                     self.opt_step(p, g['weight_decay'], lr, use_muon)
#         self.i += 1

#     def zero_grad(self):
#         for g in self.params :
#             for p in g['params'] :
#                 if p.grad is not None : p.grad.data.zero_()

#     def opt_step(self, p, wd, lr, use_muon=None):
#         if use_muon is None:
#             use_muon = (p.dim() >= 2)   # fallback for old-style groups

#         if not use_muon:
#             if not hasattr(p,'grad_avg'): p.grad_avg = torch.zeros_like(p.grad.data)
#             if not hasattr(p, 'grad_sqr_avg'): p.grad_sqr_avg = torch.zeros_like(p.grad.data)
#             p.grad_avg.lerp_(p.grad, 1 - self.beta1)
#             p.grad_sqr_avg.lerp_(p.grad.square(), 1 - self.beta2)
#             unbiased_grad_avg = p.grad_avg / (1 - self.beta1 ** (self.i+1) )
#             unbiased_grad_sqr_avg = p.grad_sqr_avg / (1 - self.beta2 ** (self.i+1) )
#             update = unbiased_grad_avg / (unbiased_grad_sqr_avg.sqrt() + self.eps)
#             if wd : update += wd * p.data
#             p.data.sub_(lr * update)
#         else:
#             g = p.grad.data

#             # Nesterov momentum
#             if not hasattr(p,'grad_avg'): p.grad_avg = torch.zeros_like(g)
#             p.grad_avg.lerp_(g, 1 - self.beta1)
#             unbiased_grad_avg = p.grad_avg  / (1 - self.beta1 ** (self.i+1) )
#             g = g.lerp(unbiased_grad_avg, self.beta1) if self.Nesterov else unbiased_grad_avg

#             # Normalization using Norm
#             target = g.norm(dim=(-1,-2), keepdim=True) * (g.size(-2)**-0.5)
#             row_norm = g.norm(dim=(-1), keepdim=True)
#             g = g * (target / (row_norm+self.eps))

#             # Frobenius norm
#             g /= (g.norm(dim=(-1,-2), keepdim=True) * 1.01 + self.eps)  # 1.01 is just safety so that everything is <1 and <-1 and not =

#             # Newton sched
#             a, b, c = 3.4445, -4.7750, 2.0315
#             for _ in range(self.steps):
#                 A = g @ g.mT                # mT transposes the last 2 dims
#                 B = b * A + c * (A @ A)
#                 g = a * g + B @ g

#             # Muon+ normalization
#             targ_norm = min(g.size(-2), g.size(-1))  ** 0.5
#             current_norm = g.norm(dim=(-1,-2), keepdim=True)
#             g = g * (targ_norm / (current_norm+self.eps))

#             # Variance Reduction
#             v_mean = g.square().mean(dim=self.red_dim, keepdim=True)
#             red_dim_sz = g.size(self.red_dim)
#             v_norm_sq = v_mean.sum(dim=(-1,-2), keepdim=True) * red_dim_sz
#             v_norm = v_norm_sq.sqrt()
#             if not hasattr(p, 'v_mean_avg'): p.v_mean_avg = torch.zeros_like(v_mean)
#             p.v_mean_avg.lerp_(v_mean, 1 - self.beta2)
#             unbiased_v_mean_avg = p.v_mean_avg  / (1 - self.beta2 ** (self.i+1))
#             step_sz = (unbiased_v_mean_avg+self.eps).rsqrt()
#             scaled_sq_sum = (v_mean * red_dim_sz) * step_sz.square()
#             v_norm_new = scaled_sq_sum.sum(dim=(-1,-2), keepdim=True).sqrt()
#             final_scale = step_sz * (v_norm / v_norm_new)
#             g = g * final_scale

#             # Update
#             mask = (g * unbiased_grad_avg) >= 0
#             update = g + wd * p.data * mask if wd else g
#             p.data.sub_(lr * update) # Make it in place


###    MAKING A MORE OPTIMIZED VERSION


@torch.compile(dynamic=False, fullgraph=True)
def _muon_step(p, grad, grad_avg, v_mean_avg, step_t, lr_t, wd_t,
               beta1: float, beta2: float, steps: int, red_dim: int, nesterov: bool, eps: float):
    g = grad.float()   # SAFETY: whole thing in fp32, cast back only at the very end

    # Nesterov momentum -- exact same as yours
    grad_avg.lerp_(g, 1 - beta1)
    bias1 = 1 - beta1 ** step_t
    unbiased_grad_avg = grad_avg / bias1
    g = g.lerp(unbiased_grad_avg, beta1) if nesterov else unbiased_grad_avg

    # row equilibration -- unchanged
    target = g.norm(dim=(-1, -2), keepdim=True) * (g.size(-2) ** -0.5)
    row_norm = g.norm(dim=-1, keepdim=True)
    g = g * (target / (row_norm + eps))

    # frobenius norm scaling -- unchanged
    g = g / (g.norm(dim=(-1, -2), keepdim=True) * 1.01 + eps)

    # newton-schulz -- your exact single (a,b,c), same steps count, unchanged
    a, b, c = 3.4445, -4.7750, 2.0315
    for _ in range(steps):
        A = g @ g.mT
        B = b * A + c * (A @ A)
        g = a * g + B @ g

    # muon+ normalization -- unchanged
    targ_norm = min(g.size(-2), g.size(-1)) ** 0.5
    current_norm = g.norm(dim=(-1, -2), keepdim=True)
    g = g * (targ_norm / (current_norm + eps))

    # variance reduction -- unchanged
    v_mean = g.square().mean(dim=red_dim, keepdim=True)
    red_dim_sz = g.size(red_dim)
    v_norm = (v_mean.sum(dim=(-1, -2), keepdim=True) * red_dim_sz).sqrt()
    v_mean_avg.lerp_(v_mean, 1 - beta2)
    bias2 = 1 - beta2 ** step_t
    unbiased_v_mean_avg = v_mean_avg / bias2
    step_sz = (unbiased_v_mean_avg + eps).rsqrt()
    scaled_sq_sum = (v_mean * red_dim_sz) * step_sz.square()
    v_norm_new = scaled_sq_sum.sum(dim=(-1, -2), keepdim=True).sqrt()
    final_scale = step_sz * (v_norm / (v_norm_new + eps))
    g = g * final_scale

    # cautious weight decay + update -- unchanged
    mask = (g * unbiased_grad_avg) >= 0
    update = g + wd_t * p.float() * mask
    p.sub_((lr_t * update).to(p.dtype))


@torch.compile(dynamic=False, fullgraph=True)
def _adamw_step(p, grad, exp_avg, exp_avg_sq, step_t, lr_t, wd_t, beta1: float, beta2: float, eps: float):
    g = grad.float()
    exp_avg.lerp_(g, 1 - beta1)
    exp_avg_sq.lerp_(g.square(), 1 - beta2)
    bias1 = 1 - beta1 ** step_t
    bias2 = 1 - beta2 ** step_t
    update = (exp_avg / bias1) / ((exp_avg_sq / bias2).sqrt() + eps)
    update = update + wd_t * p.float()          # same as  `update += wd*p.data`, just always-on (wd_t=0 if unused)
    p.sub_((lr_t * update).to(p.dtype))


class MuonAdamW:
    def __init__(self, params, lr, beta2, red_dim, steps=5, beta1=0.9, Nesterov=True, eps=1e-7):
        self.params = params
        self.lr = lr
        self.beta2 = beta2
        self.red_dim = red_dim
        self.steps = steps
        self.beta1 = beta1
        self.Nesterov = Nesterov
        self.eps = eps
        self.i = 0
        self.state = {}                                       # FIX: real state, not p.attribute
        self._lr_t = torch.zeros((), dtype=torch.float32)      # FIX: reused tensors, no recompile storm
        self._wd_t = torch.zeros((), dtype=torch.float32)
        self._step_t = torch.zeros((), dtype=torch.float32)

    def step(self, lr=None):
        if lr is None: lr = self.lr
        self.i += 1
        with torch.no_grad():
            for g in self.params:
                use_muon = g.get('use_muon', None)
                for p in g['params']:
                    if p.grad is None:                         # FIX: don't crash on missing grads
                        continue
                    self.opt_step(p, g['weight_decay'], lr, use_muon)

    def zero_grad(self):
        for g in self.params:
            for p in g['params']:
                if p.grad is not None: p.grad.data.zero_()

    def opt_step(self, p, wd, lr, use_muon=None):
        if use_muon is None:
            use_muon = (p.dim() >= 2)  # fallback for old-style groups

        self._lr_t.fill_(lr)
        self._wd_t.fill_(wd or 0.0)
        self._step_t.fill_(float(self.i))

        state = self.state.setdefault(p, {})

        if not use_muon:
            if 'exp_avg' not in state:
                state['exp_avg'] = torch.zeros_like(p, dtype=torch.float32)     # FIX: fp32 even if p is bf16
                state['exp_avg_sq'] = torch.zeros_like(p, dtype=torch.float32)
            _adamw_step(p.data, p.grad.data, state['exp_avg'], state['exp_avg_sq'],
                        self._step_t, self._lr_t, self._wd_t, self.beta1, self.beta2, self.eps)
        else:
            if 'grad_avg' not in state:
                state['grad_avg'] = torch.zeros_like(p, dtype=torch.float32)    # FIX: fp32 even if p is bf16
                v_shape = list(p.shape)
                v_shape[self.red_dim] = 1
                state['v_mean_avg'] = torch.zeros(v_shape, dtype=torch.float32, device=p.device)
            _muon_step(p.data, p.grad.data, state['grad_avg'], state['v_mean_avg'],
                       self._step_t, self._lr_t, self._wd_t,
                       self.beta1, self.beta2, self.steps, self.red_dim, self.Nesterov, self.eps)


###_________________________ CREATING THE MODEL _______________________________________

def batch_head(x, n_head):
  b,t,c = x.shape
  x = x.reshape(b, t, n_head,-1)
  return x.transpose(1,2).reshape(b*n_head, t, -1)

def head_batch(x, n_head):
    bn, t, c = x.shape
    x = x.reshape(-1, n_head, t, c)
    return x.transpose(1, 2).reshape(-1, t, n_head * c)


class CausalMultiHeadAttention(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.n_head = config.n_head
    self.n_embd = config.n_embd
    assert self.n_embd % config.n_head == 0, "number of heads must be a multiple of n_head"
    self.head_sz = config.n_embd // config.n_head
    self.c_attn = nn.Linear(self.n_embd, 3 * self.n_embd)
    self.c_proj = nn.Linear(self.n_embd,self.n_embd)
    self.c_proj.INIT_SPECIAL_STD = 1
    # self.register_buffer('bias', torch.tril(torch.ones(config.block_size, config.block_size)))   # -> because we switched to Flash attention

  def forward(self, x): # x -> (B, T, E)
    b,t,c = x.shape
    x = self.c_attn(x)
    q, k, v = torch.chunk(x, 3, dim=-1) # we get the qkv first before batch_head because this is the order openai used(weights optimized for it), but reversing it when training a model from scratch is fine
    q = batch_head(q, self.n_head) # (B, T, E)
    k = batch_head(k, self.n_head) # (B, T, E)
    v = batch_head(v, self.n_head) # (B, T, E)
    # wei = q @ k.transpose(-1,-2) * self.head_sz**-0.5 # Scaling, shape (B, T, T)
    # wei = wei.masked_fill(self.bias[:t,:t]==0, float('-inf'))
    # wei = F.softmax(wei, -1)
    # wei = wei @ v
    wei = F.scaled_dot_product_attention(q, k, v, is_causal=True) # is_casual=True -> causal mask is the triangular mask we built manually with torch.tril + masked_fill.
    wei = head_batch(wei, self.n_head)
    return self.c_proj(wei)


class MLP(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd)
    self.gelu    = nn.GELU(approximate='tanh')
    self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd)
    self.c_proj.INIT_SPECIAL_STD = 1
  def forward(self, x):
    return self.c_proj(self.gelu(self.c_fc(x)))



class Block(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.attn = CausalMultiHeadAttention(config)
    self.mlp = MLP(config)
    self.ln_1 = nn.LayerNorm(config.n_embd)
    self.ln_2 = nn.LayerNorm(config.n_embd)
  def forward(self, x):
    x = x + self.attn(self.ln_1(x))
    return x + self.mlp(self.ln_2(x))


class GPT(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.config = config
    self.transformer = nn.ModuleDict(dict(
      wpe = nn.Embedding(config.block_size, config.n_embd),
      wte = nn.Embedding(config.vocab_size, config.n_embd),
      h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
      ln_f = nn.LayerNorm(config.n_embd)
    ))
    self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

    ## MAKE the last Linear later the same as the embedding layer
    self.lm_head.weight = self.transformer.wte.weight

    ## APPLY Initialization
    self.apply(self._initalization)

  ## Make Initialization function
  def _initalization(self, module):
      if isinstance(module, nn.Linear):
          std = 0.02
          if hasattr(module, 'INIT_SPECIAL_STD'):  std *= (2 * self.config.n_layer) ** -0.5
          nn.init.normal_(module.weight,mean=0., std=std) # torch.nn
          if module.bias is not None:  nn.init.zeros_(module.bias)
      elif isinstance(module, nn.Embedding):
          nn.init.normal_(module.weight, mean=0., std=0.02)

  def forward(self, x, targets=None): # x -> (B, Tokens)
    B, T = x.shape
    assert T <= self.config.block_size, f'The Context Exceeds the block size {T} > {self.config.block_size}'

    x = self.transformer.wpe(torch.arange(T, device=x.device)) + self.transformer.wte(x) # x -> (B, tokens, embs)\

    for layer in self.transformer.h: x = layer(x)

    x = self.transformer.ln_f(x)

    logits = self.lm_head(x)
    loss = None
    if targets is not None:  loss = F.cross_entropy(logits.view(-1, logits.shape[-1]), targets.view(-1))
    return logits, loss

  @classmethod
  def from_pretrained(cls, model_name): #  cls -> the class it self can be called -> cls()
    assert model_name in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}, f"{model_name} is not Valid" # make a set so that it is faster O(1)
    from transformers import GPT2LMHeadModel
    print(f'loading {model_name}')
    configs ={
        'gpt2':         dict(n_layer=12, n_head=12, n_embd=768),  # 124M params
        'gpt2-medium':  dict(n_layer=24, n_head=16, n_embd=1024), # 350M params
        'gpt2-large':   dict(n_layer=36, n_head=20, n_embd=1280), # 774M params
        'gpt2-xl':      dict(n_layer=48, n_head=25, n_embd=1600), # 1558M params
    }[model_name]
    configs['vocab_size'] = 50257 # same in all of them
    configs['block_size'] = 1024 # same in all of them

    model_config = GPT_config(**configs)
    model = cls(model_config)
    sd = model.state_dict()
    sd_keys = sd.keys()
    sd_keys = [k for k in sd_keys if not k.endswith('.attn.bias')] # discard this mask / buffer, not a param

    model_hf = GPT2LMHeadModel.from_pretrained(model_name) # load the hugging face model
    sd_hf = model_hf.state_dict()
    sd_keys_hf = sd_hf.keys()
    sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.masked_bias')] # ignore these, just a buffer
    sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.bias')] # same, just the mask (buffer)
    transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight'] # openai checkpoints use a "Conv1D" module, but we only want to use a  Linear so we have to transpose these weights when we import them

    assert len(sd_keys) == len(sd_keys_hf), f'Keys mismatch : {len(sd_keys_hf)} != {len(sd_keys)}'
    for k in sd_keys_hf:
        if any(k.endswith(w) for w in transposed):
            assert sd_hf[k].shape[::-1] == sd[k].shape, f'shape mismatch Transposed if Conv-> {sd_hf[k].shape} != {sd[k].shape} at {k}'   # [::-1] reverse sequence e.g. (768, 3072) to (3072, 768)
            with torch.no_grad():
                sd[k].copy_(sd_hf[k].T)
        else:
            assert sd_hf[k].shape == sd[k].shape , f'shape mismatch {sd_hf[k].shape} != {sd[k].shape} at {k}'
            with torch.no_grad():
                sd[k].copy_(sd_hf[k])

    return model

  def optimizers_config(self, device_type, optimizer=None):
    params_dict = {pn: p for pn, p in self.named_parameters() if p.requires_grad}

    # Embedding / unembedding must NOT go through Muon's Newton-Schulz step
    no_muon_names = {'transformer.wte.weight', 'lm_head.weight'}

    if optimizer is None:
        decay_params = [p for pn, p in params_dict.items() if p.dim() >= 2]
        no_decay_params = [p for pn, p in params_dict.items() if p.dim() < 2]
        optim_group = [
            {'params': decay_params, 'weight_decay': self.config.weight_decay},
            {'params': no_decay_params, 'weight_decay': 0.},
        ]
        fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
        use_fuse = fused_available and device_type == 'cuda'
        return torch.optim.AdamW(optim_group, lr=self.config.max_lr,
                                  betas=(self.config.beta1, self.config.beta2),
                                  eps=self.config.eps, fused=use_fuse)
    else:
        muon_params = [p for pn, p in params_dict.items()
                        if p.dim() >= 2 and pn not in no_muon_names]
        adam_params = [p for pn, p in params_dict.items()
                        if p.dim() < 2 or pn in no_muon_names]
        optim_group = [
            {'params': muon_params, 'weight_decay': self.config.weight_decay, 'use_muon': True},
            {'params': adam_params, 'weight_decay': 0.,                       'use_muon': False},
        ]
        return optimizer(optim_group, lr=self.config.max_lr, beta1=self.config.beta1,
                          beta2=self.config.beta2, eps=self.config.eps,
                          steps=self.config.steps, red_dim=self.config.red_dim,
                          Nesterov=self.config.Nesterov)

###______________________________________ MAKE A DATALOADER ________________________________

def load_tokens(filename):
    np_loaded = np.load(filename)
    np_loaded = np_loaded.astype(np.int32) # convert uint16 to int32 before cast to long, otherwise pytorch doesn't like it
    return torch.tensor(np_loaded, dtype=torch.long)




# Make DL lite
class DL_lite:
    def __init__(self, config, split):
        assert split in {'Train', 'Val'}, 'Split must be one of "Train" or "Val"'
        # store attr
        self.block_size= config.block_size
        self.process_rank = config.process_rank
        self.num_processes = config.num_processes
        self.bs = config.bs
        assert config.tot_bs_for_grad_accum % (self.bs * config.num_processes) == 0, f' Total_Batch_size: {config.tot_bs_for_grad_accum} is not divisible by Batch_size: {self.bs} * Num_processes: {self.num_processes}'
        grad_accum = bool(config.tot_bs_for_grad_accum)
        self.tot_mini_batches = config.tot_bs_for_grad_accum // (self.bs * self.block_size * self.num_processes) if grad_accum else 1
        # Loading the data
        data_root = config.data_root
        shards = os.listdir(data_root)
        shards = [s for s in shards if split in s]
        shards = sorted(shards)
        self.shards = [os.path.join(data_root, s) for s in shards]
        assert len(shards) > 0, f"no shards found for split {split}"
        if master_process: print(f'Found Shards =  {len(self.shards)} | Split = {split} | Total Batch Size = {config.tot_bs_for_grad_accum} | Grad Accum = {grad_accum} | Number Mini Batches = {self.tot_mini_batches} | Num_processes: {self.num_processes}')
        # Make a trackers
        self.reset()

    def reset(self):
        self.current_shard = 0
        self.tokens = load_tokens(self.shards[self.current_shard])
        self.tr = self.bs * self.block_size * self.process_rank

    def after_batch(self):
        tokens = self.tokens[self.tr : self.tr + self.bs * self.block_size + 1 ]
        self.tr += self.bs * self.block_size * self.num_processes
        x = tokens[:-1].view(self.bs, -1)
        y = tokens[1:].view(self.bs, -1)
        if (self.bs * self.block_size * self.num_processes  + 1 + self.tr) > len(self.tokens):
            self.current_shard = (self.current_shard + 1) % len(self.shards) # So that we advance to the next shard and if we finish the shards we loop again because of -> %
            self.tokens = load_tokens(self.shards[self.current_shard])
            self.tr = self.bs * self.block_size * self.process_rank
        return x, y

###_________________________________ HELLASWAG FUNCTION _______________written in HellaSwag.py________________________________


def get_most_likely_row(tokens, mask, logits):
    shift_logits = (logits[:, :-1, :]).contiguous()
    shift_tokens = (tokens[..., 1:]).contiguous()
    shift_logits = shift_logits.view(-1, shift_logits.size(-1))
    shift_tokens = shift_tokens.view(-1)
    loss = F.cross_entropy(shift_logits, shift_tokens, reduction='none')
    reshaped_loss = loss.view(tokens.size(0), -1)
    shift_mask = mask[..., 1:].contiguous()
    masked_shift_loss = reshaped_loss * shift_mask
    sum_loss = torch.sum(masked_shift_loss, dim=-1)
    avg_loss = sum_loss / shift_mask.sum(-1)
    norm_preds = torch.argmin(avg_loss).item()

    return norm_preds

###_________________________________ LR SCHED _______________________________________________

# Creating a LR sched
def get_lr(step, config):
    min_lr = config.max_lr * config.min_lr_ratio
    if step < config.warmup_steps: return config.max_lr/config.warmup_steps * (step+1)
    if step > config.max_steps : return min_lr
    decay_ratio = (step-config.warmup_steps)/(config.max_steps-config.warmup_steps) # make a ratio between 0 and 1 that represent the current section in the cos
    assert 0 <= decay_ratio <= 1, f'There is something wrong with max steps:{config.max_steps}, step:{step}, warmup_steps:{config.warmup_steps}'
    coeff = 0.5 * (1 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (config.max_lr - min_lr)


###_____________________________________  INSTANCES  ______________________________________

# It a PyTorch function that speeds up float32 matrix multiplications on compatible NVIDIA GPUs by trading off a small amount of numerical precision for significant performance gains.
torch.set_float32_matmul_precision('high')

# config
config = GPT_config()

# make device_type
device_type = "cuda" if config.device.startswith("cuda") else "cpu" # just to use it at the autocast, etc... and make 'cuda:3' -> 'cuda', 'cuda' -> 'cuda' ,etc...

# encoder
enc = tiktoken.get_encoding(config.tokenizer)

# model
model = GPT(config)
model.to(config.device)
if config.use_compile: model = torch.compile(model) # compiles the model and makes kernel fusion for the operations
if ddp :  model = DDP(model, device_ids=[ddp_local_rank]) # Forward pass / training step → use model (the DDP wrapper) — this is what makes multi-GPU synchronization work
raw_model = model.module if ddp else model  # always contains the "raw" unwrapped model / -> Anything else (custom methods, saving checkpoints, accessing .config) → use raw_model — because DDP's wrapper doesn't expose your class's custom stuff directly

# lr getter fn
lr_getter = partial(get_lr, config=config)

# data
train_dl = DL_lite(config, 'Train')
val_dl = DL_lite(config, 'Val')

# optim
optimizer = raw_model.optimizers_config(device_type, MuonAdamW)

# make a logging file
log_dir = 'log'
os.makedirs(log_dir, exist_ok=True)
log_file = os.path.join(log_dir, 'log.txt')
with open(log_file, 'w') as f: # open for writing to clear the file
    pass

##_____________________________________  TRAINING  ______________________________________


# Training Loop
for step in range(config.training_steps):
    start = time.time()
    last_step = (step == config.training_steps - 1)

    # Validation
    if (step % config.val_after_step == 0 or last_step) and (config.validation) :
        model.eval()
        with torch.no_grad():
            val_accum_loss = 0.
            for _ in range(config.val_loss_accum_steps):
                x, y = val_dl.after_batch()
                x, y = x.to(config.device), y.to(config.device)
                with torch.autocast(device_type=device_type, dtype=torch.bfloat16):
                    logits, loss = model(x, y)
                loss /= config.val_loss_accum_steps
                val_accum_loss += loss.detach()
        if ddp: dist.all_reduce(val_accum_loss, op=dist.ReduceOp.AVG)
        val_dl.reset()
        if master_process:
            print(f"Validation loss: {val_accum_loss.item():.4f}")
            with open(log_file, "a") as f: f.write(f"{step} val {val_accum_loss.item():.4f}\n")

            # Save checkpoints for the model
            if (step > 0) and (step % config.checkpoint_after_steps == 0 or last_step):
                checkpoint_path = os.path.join(log_dir, f'model_checkpoint_step_{step:05d}.pt')
                checkpoint = {
                    'model': raw_model.state_dict(),
                    'config': raw_model.config, # that is the stored config in the model object as an attr
                    'step': step,
                    'val_loss': val_accum_loss.item()
                }
                torch.save(checkpoint, checkpoint_path)


    # Model Sampling
    if step % config.val_after_step == 0 and step > 0 and config.model_sampling and (not config.use_compile):
        model.eval()
        tokens = enc.encode('I am crazy man,')
        tokens = torch.tensor(tokens, dtype=torch.long)
        tokens = tokens.repeat(config.num_sequence, 1)
        x_gen = tokens.to(config.device)

        while x_gen.shape[1] < config.max_length:
            with torch.no_grad():
                with torch.autocast(device_type=device_type, dtype=torch.bfloat16):
                    logits, loss = model(x_gen)
                logits = logits[:, -1, :]
                probs = F.softmax(logits, dim=-1)
                topk_probs, topk_indices = torch.topk(probs, k=50, dim=-1)  # take the highest 50 probs
                ix = torch.multinomial(topk_probs, num_samples=1)  # take one sample
                ids = torch.gather(topk_indices, dim=-1, index=ix)
                x_gen = torch.cat((x_gen, ids), dim=1)

        for i in range(config.num_sequence):
            decoded = enc.decode(x_gen[i, :config.max_length].tolist())
            print(f'Rank: {config.process_rank} | Sample{i + 1}: {decoded} ')

    # HellaSwag
    # once in a while evaluate hellaswag
    if (step % config.val_after_step == 0 or last_step) and (not config.use_compile):
        num_correct_norm = 0
        num_total = 0
        for i, example in enumerate(iterate_examples("val")):
            # only process examples where i % ddp_world_size == ddp_rank
            if i % ddp_world_size != ddp_rank:  continue
            # render the example into tokens and labels
            _, tokens, mask, label = render_example(example)
            tokens = tokens.to(device)
            mask = mask.to(device)
            # get the logits
            with torch.no_grad():
                with torch.autocast(device_type=device_type, dtype=torch.bfloat16):
                    logits, loss = model(tokens)
                pred_norm = get_most_likely_row(tokens, mask, logits)
            num_total += 1
            num_correct_norm += int(pred_norm == label)
        # reduce the stats across all processes
        if ddp:
            num_total = torch.tensor(num_total, dtype=torch.long, device=device)
            num_correct_norm = torch.tensor(num_correct_norm, dtype=torch.long, device=device)
            dist.all_reduce(num_total, op=dist.ReduceOp.SUM)
            dist.all_reduce(num_correct_norm, op=dist.ReduceOp.SUM)
            num_total = num_total.item()
            num_correct_norm = num_correct_norm.item()
        acc_norm = num_correct_norm / num_total
        if master_process:
            print(f"HellaSwag accuracy: {num_correct_norm}/{num_total}={acc_norm:.4f}")
            with open(log_file, "a") as f:  f.write(f"{step} hella {acc_norm:.4f}\n")


    # Training
    model.train()
    accum_loss = 0
    for mini_step in range(train_dl.tot_mini_batches):
        x,y = train_dl.after_batch()
        x,y = x.to(config.device), y.to(config.device)
        if ddp:  model.require_backward_grad_sync = (mini_step == train_dl.tot_mini_batches - 1) # So that we avoid unnecessary communication during backward () unless it is the last step
        with torch.autocast(device_type=device_type, dtype=torch.bfloat16):
            logits, loss = model(x, y)
            loss /= train_dl.tot_mini_batches
            accum_loss += loss.detach()
        loss.backward()
    if ddp:  dist.all_reduce(accum_loss, op= dist.ReduceOp.AVG) # Averaging the Loss Across all the Processes
    lr = lr_getter(step)
    # for param_group in optimizer.param_groups:  -----> Becasue we are using the MuonAdamW optimizer, we don't need to set the lr for each param group, we just pass it to the step function
    #     param_group['lr'] = lr
    norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.) # This Makes Normalization for the Norm of the Gradients proportionally, so that -> root(sum(grads**2)) <= 1
    optimizer.step(lr)
    optimizer.zero_grad()
    if device_type == 'cuda' : torch.cuda.synchronize() # so that the cpu don't run the next command while the GPU still hasn't Finished

    # Printings
    end = time.time()
    time_taken = end - start
    tokens_count = train_dl.block_size * train_dl.bs * train_dl.tot_mini_batches
    if master_process:
        print(f'Step: {step:5d} | Loss: {accum_loss.item():.4f} | lr = {lr:.6f} | Norm = {norm:6f} | Time: {time_taken:.4f}sec | Token/sec: {(tokens_count / time_taken):.3f}')
        with open(log_file, 'a') as f: f.write(f"{step} train {accum_loss.item():.6f}\n")

if ddp: destroy_process_group() # Clean After the Multi-GPU Process




Writing train_gpt2.py


In [3]:
!cd /kaggle/working

In [4]:
!rm -rf /kaggle/working/HellaSwag

In [5]:
!ls

HellaSwag.py  train_gpt2.py


In [ ]:
!torchrun --standalone --nproc_per_node=2 train_gpt2.py

W0715 18:34:57.790000 117 torch/distributed/run.py:852] 
W0715 18:34:57.790000 117 torch/distributed/run.py:852] *****************************************
W0715 18:34:57.790000 117 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0715 18:34:57.790000 117 torch/distributed/run.py:852] *****************************************
[W715 18:34:58.907893498 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W715 18:35:24.141673660 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W715 18:35:24.294618622 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
Found Shards =  2 | Split = Train | Total Batch Size = 131072 | Grad Accum = True | Number Mini Batches = 8 | Num_processes: 2
Found Shards =  1 | Spli


    W0715 18:34:57.790000 117 torch/distributed/run.py:852]
    W0715 18:34:57.790000 117 torch/distributed/run.py:852] *****************************************
    W0715 18:34:57.790000 117 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed.
    W0715 18:34:57.790000 117 torch/distributed/run.py:852] *****************************************
    [W715 18:34:58.907893498 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
    [W715 18:35:24.141673660 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
    [W715 18:35:24.294618622 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
    Found Shards =  2 | Split = Train | Total Batch Size = 131072 | Grad Accum = True | Number Mini Batches = 8 | Num_processes: 2
    Found Shards =  1 | Split = Val | Total Batch Size = 131072 | Grad Accum = True | Number Mini Batches = 8 | Num_processes: 2
    Downloading val data from https://raw.githubusercontent.com/rowanz/hellaswag/master/data/hellaswag_val.jsonl to /kaggle/working/HellaSwag/hellaswag_val.jsonl
    Validation loss: 10.9798
    Downloading val data from https://raw.githubusercontent.com/rowanz/hellaswag/master/data/hellaswag_val.jsonl to /kaggle/working/HellaSwag/hellaswag_val.jsonl
    /kaggle/working/HellaSwag/hellaswag_val.jsonl: 11.7MiB [00:00, 68.7MiB/s]
    /kaggle/working/HellaSwag/hellaswag_val.jsonl: 11.7MiB [00:00, 70.2MiB/s]
    HellaSwag accuracy: 2509/10042=0.2499
    [rank0]:W0715 18:41:23.233000 125 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode
    [rank1]:W0715 18:41:23.234000 126 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode
    Step:     0 | Loss: 10.9814 | lr = 0.000060 | Norm = 13.880294 | Time: 374.1474sec | Token/sec: 175.161
    Step:     1 | Loss: 10.9079 | lr = 0.000120 | Norm = 14.436560 | Time: 39.8969sec | Token/sec: 1642.635
    Step:     2 | Loss: 10.7725 | lr = 0.000180 | Norm = 13.962316 | Time: 37.6986sec | Token/sec: 1738.422
    Step:     3 | Loss: 10.5472 | lr = 0.000240 | Norm = 13.428261 | Time: 38.1902sec | Token/sec: 1716.043
    Step:     4 | Loss: 10.2806 | lr = 0.000300 | Norm = 12.444089 | Time: 38.1415sec | Token/sec: 1718.233
    Step:     5 | Loss: 9.9928 | lr = 0.000360 | Norm = 11.026724 | Time: 37.9373sec | Token/sec: 1727.483
    Step:     6 | Loss: 9.5639 | lr = 0.000420 | Norm = 10.705714 | Time: 38.1305sec | Token/sec: 1718.727
    Step:     7 | Loss: 9.1847 | lr = 0.000480 | Norm = 9.679224 | Time: 38.1801sec | Token/sec: 1716.499
    Step:     8 | Loss: 8.7697 | lr = 0.000540 | Norm = 8.544680 | Time: 37.9755sec | Token/sec: 1725.746
    Step:     9 | Loss: 8.3983 | lr = 0.000600 | Norm = 6.906669 | Time: 37.8982sec | Token/sec: 1729.263
    Step:    10 | Loss: 8.1187 | lr = 0.000600 | Norm = 6.080722 | Time: 37.7982sec | Token/sec: 1733.839
    Step:    11 | Loss: 7.9271 | lr = 0.000600 | Norm = 6.407850 | Time: 37.7644sec | Token/sec: 1735.393
    Step:    12 | Loss: 7.7924 | lr = 0.000600 | Norm = 5.982113 | Time: 37.7100sec | Token/sec: 1737.894
    Step:    13 | Loss: 7.7619 | lr = 0.000600 | Norm = 5.252160 | Time: 37.6875sec | Token/sec: 1738.930
    Step:    14 | Loss: 7.7209 | lr = 0.000600 | Norm = 4.319501 | Time: 37.7515sec | Token/sec: 1735.983
    Step:    15 | Loss: 7.7157 | lr = 0.000600 | Norm = 3.919659 | Time: 37.7559sec | Token/sec: 1735.784
    Step:    16 | Loss: 7.6871 | lr = 0.000600 | Norm = 4.030606 | Time: 37.7764sec | Token/sec: 1734.841
    Step:    17 | Loss: 7.7316 | lr = 0.000600 | Norm = 4.039842 | Time: 37.8241sec | Token/sec: 1732.654
    Step:    18 | Loss: 7.7064 | lr = 0.000600 | Norm = 3.935467 | Time: 38.0636sec | Token/sec: 1721.750
    Step:    19 | Loss: 7.5537 | lr = 0.000600 | Norm = 2.982925 | Time: 37.8058sec | Token/sec: 1733.493
    Step:    20 | Loss: 7.6591 | lr = 0.000600 | Norm = 2.693980 | Time: 37.6358sec | Token/sec: 1741.321
    Step:    21 | Loss: 7.5988 | lr = 0.000600 | Norm = 1.867970 | Time: 38.1362sec | Token/sec: 1718.470
    Step:    22 | Loss: 7.5670 | lr = 0.000600 | Norm = 2.612561 | Time: 37.9143sec | Token/sec: 1728.528
    Step:    23 | Loss: 7.7736 | lr = 0.000600 | Norm = 2.435753 | Time: 37.6897sec | Token/sec: 1738.833
    Step:    24 | Loss: 7.7237 | lr = 0.000600 | Norm = 2.772653 | Time: 37.5604sec | Token/sec: 1744.817
    Step:    25 | Loss: 7.4624 | lr = 0.000600 | Norm = 3.036376 | Time: 37.7766sec | Token/sec: 1734.831
    Step:    26 | Loss: 7.5626 | lr = 0.000600 | Norm = 2.272755 | Time: 37.8611sec | Token/sec: 1730.957
    Step:    27 | Loss: 7.5129 | lr = 0.000600 | Norm = 2.024063 | Time: 37.8565sec | Token/sec: 1731.168
    Step:    28 | Loss: 7.5067 | lr = 0.000599 | Norm = 2.366513 | Time: 37.7930sec | Token/sec: 1734.076
    Step:    29 | Loss: 7.5503 | lr = 0.000599 | Norm = 2.501288 | Time: 37.7666sec | Token/sec: 1735.291
    Step:    30 | Loss: 7.3854 | lr = 0.000599 | Norm = 2.578380 | Time: 37.7927sec | Token/sec: 1734.092
    Step:    31 | Loss: 7.4173 | lr = 0.000599 | Norm = 2.496969 | Time: 37.8642sec | Token/sec: 1730.817
    Step:    32 | Loss: 7.3626 | lr = 0.000599 | Norm = 2.340298 | Time: 37.9532sec | Token/sec: 1726.758
    Step:    33 | Loss: 7.2882 | lr = 0.000599 | Norm = 2.438547 | Time: 37.9589sec | Token/sec: 1726.500
    Step:    34 | Loss: 7.2812 | lr = 0.000599 | Norm = 2.520598 | Time: 37.8447sec | Token/sec: 1731.709
    Step:    35 | Loss: 7.2269 | lr = 0.000599 | Norm = 3.078116 | Time: 37.7553sec | Token/sec: 1735.811
    Step:    36 | Loss: 7.1230 | lr = 0.000599 | Norm = 2.524240 | Time: 37.7230sec | Token/sec: 1737.293
    Step:    37 | Loss: 7.1344 | lr = 0.000599 | Norm = 2.405705 | Time: 37.8566sec | Token/sec: 1731.162
    Step:    38 | Loss: 7.0850 | lr = 0.000599 | Norm = 2.518343 | Time: 38.0967sec | Token/sec: 1720.253
    Step:    39 | Loss: 7.0924 | lr = 0.000599 | Norm = 2.497257 | Time: 38.3347sec | Token/sec: 1709.575
    Step:    40 | Loss: 7.0959 | lr = 0.000598 | Norm = 2.430627 | Time: 38.0379sec | Token/sec: 1722.913
    Step:    41 | Loss: 7.0345 | lr = 0.000598 | Norm = 1.607716 | Time: 37.9735sec | Token/sec: 1725.835
    Step:    42 | Loss: 6.9751 | lr = 0.000598 | Norm = 2.016786 | Time: 37.9314sec | Token/sec: 1727.750
    Step:    43 | Loss: 6.9485 | lr = 0.000598 | Norm = 1.906907 | Time: 37.8509sec | Token/sec: 1731.427
    Step:    44 | Loss: 6.9541 | lr = 0.000598 | Norm = 1.647979 | Time: 38.1348sec | Token/sec: 1718.535
    Step:    45 | Loss: 6.8160 | lr = 0.000598 | Norm = 2.118807 | Time: 38.3623sec | Token/sec: 1708.343
    Step:    46 | Loss: 6.8460 | lr = 0.000598 | Norm = 2.181834 | Time: 38.0729sec | Token/sec: 1721.329
    Step:    47 | Loss: 6.8137 | lr = 0.000598 | Norm = 2.612846 | Time: 37.8167sec | Token/sec: 1732.990
    Step:    48 | Loss: 6.7763 | lr = 0.000598 | Norm = 2.008649 | Time: 37.8541sec | Token/sec: 1731.281
    Step:    49 | Loss: 6.7701 | lr = 0.000597 | Norm = 1.706160 | Time: 37.8334sec | Token/sec: 1732.227
    Step:    50 | Loss: 6.8618 | lr = 0.000597 | Norm = 2.172903 | Time: 37.7542sec | Token/sec: 1735.862
    Step:    51 | Loss: 6.9037 | lr = 0.000597 | Norm = 3.107383 | Time: 38.0944sec | Token/sec: 1720.358
    Step:    52 | Loss: 6.7049 | lr = 0.000597 | Norm = 2.012556 | Time: 38.3810sec | Token/sec: 1707.510
    Step:    53 | Loss: 6.6495 | lr = 0.000597 | Norm = 2.248986 | Time: 38.1717sec | Token/sec: 1716.875
    Step:    54 | Loss: 6.7619 | lr = 0.000597 | Norm = 1.973611 | Time: 37.7504sec | Token/sec: 1736.036
    Step:    55 | Loss: 6.8220 | lr = 0.000597 | Norm = 2.269523 | Time: 37.7727sec | Token/sec: 1735.007
    Step:    56 | Loss: 6.6085 | lr = 0.000596 | Norm = 3.182795 | Time: 37.8058sec | Token/sec: 1733.490
    Step:    57 | Loss: 6.5326 | lr = 0.000596 | Norm = 1.992892 | Time: 37.8000sec | Token/sec: 1733.756
    Step:    58 | Loss: 6.5696 | lr = 0.000596 | Norm = 2.601860 | Time: 37.7858sec | Token/sec: 1734.408
    Step:    59 | Loss: 6.3812 | lr = 0.000596 | Norm = 4.312301 | Time: 38.1214sec | Token/sec: 1719.137
    Step:    60 | Loss: 6.5777 | lr = 0.000596 | Norm = 2.639452 | Time: 38.2695sec | Token/sec: 1712.488
    Step:    61 | Loss: 6.5025 | lr = 0.000596 | Norm = 2.900787 | Time: 38.2736sec | Token/sec: 1712.305
    Step:    62 | Loss: 6.5924 | lr = 0.000595 | Norm = 5.435210 | Time: 38.1663sec | Token/sec: 1717.117
    Step:    63 | Loss: 6.5340 | lr = 0.000595 | Norm = 4.631206 | Time: 38.1326sec | Token/sec: 1718.634
    Step:    64 | Loss: 6.4651 | lr = 0.000595 | Norm = 2.407893 | Time: 38.1334sec | Token/sec: 1718.599
    Step:    65 | Loss: 6.5460 | lr = 0.000595 | Norm = 2.103162 | Time: 38.2398sec | Token/sec: 1713.817
    Step:    66 | Loss: 6.5757 | lr = 0.000595 | Norm = 2.297327 | Time: 38.1418sec | Token/sec: 1718.220
    Step:    67 | Loss: 6.5046 | lr = 0.000595 | Norm = 2.852317 | Time: 37.9737sec | Token/sec: 1725.824
    Step:    68 | Loss: 6.5439 | lr = 0.000594 | Norm = 2.830438 | Time: 37.8973sec | Token/sec: 1729.304
    Step:    69 | Loss: 6.4179 | lr = 0.000594 | Norm = 3.653618 | Time: 37.8196sec | Token/sec: 1732.858
    Step:    70 | Loss: 6.4210 | lr = 0.000594 | Norm = 3.201266 | Time: 37.8008sec | Token/sec: 1733.718
    Step:    71 | Loss: 6.3763 | lr = 0.000594 | Norm = 2.829335 | Time: 37.7830sec | Token/sec: 1734.538
    Step:    72 | Loss: 6.4056 | lr = 0.000594 | Norm = 3.296696 | Time: 37.8256sec | Token/sec: 1732.584
    Step:    73 | Loss: 6.5380 | lr = 0.000593 | Norm = 4.203443 | Time: 37.8144sec | Token/sec: 1733.097
    Step:    74 | Loss: 6.4360 | lr = 0.000593 | Norm = 2.103035 | Time: 37.8422sec | Token/sec: 1731.822
    Step:    75 | Loss: 7.0987 | lr = 0.000593 | Norm = 5.426840 | Time: 37.8707sec | Token/sec: 1730.518
    Step:    76 | Loss: 6.3528 | lr = 0.000593 | Norm = 4.649236 | Time: 37.9308sec | Token/sec: 1727.777
    Step:    77 | Loss: 6.9525 | lr = 0.000592 | Norm = 12.159565 | Time: 37.9106sec | Token/sec: 1728.698
    Step:    78 | Loss: 6.5521 | lr = 0.000592 | Norm = 6.179507 | Time: 37.9644sec | Token/sec: 1726.247
    Step:    79 | Loss: 6.2949 | lr = 0.000592 | Norm = 2.495130 | Time: 38.0370sec | Token/sec: 1722.954
    Step:    80 | Loss: 6.3292 | lr = 0.000592 | Norm = 1.831163 | Time: 38.0357sec | Token/sec: 1723.013
    Step:    81 | Loss: 6.3462 | lr = 0.000592 | Norm = 2.114474 | Time: 38.0304sec | Token/sec: 1723.254
    Step:    82 | Loss: 6.3152 | lr = 0.000591 | Norm = 2.444216 | Time: 38.0318sec | Token/sec: 1723.187
    Step:    83 | Loss: 6.3156 | lr = 0.000591 | Norm = 3.091716 | Time: 37.9836sec | Token/sec: 1725.374
    Step:    84 | Loss: 6.3119 | lr = 0.000591 | Norm = 3.319679 | Time: 37.9494sec | Token/sec: 1726.931
    Step:    85 | Loss: 6.2699 | lr = 0.000591 | Norm = 2.960108 | Time: 37.9648sec | Token/sec: 1726.232
    Step:    86 | Loss: 6.2453 | lr = 0.000590 | Norm = 3.529890 | Time: 37.9727sec | Token/sec: 1725.870
    Step:    87 | Loss: 6.2453 | lr = 0.000590 | Norm = 3.633152 | Time: 38.0165sec | Token/sec: 1723.884
    Step:    88 | Loss: 6.2919 | lr = 0.000590 | Norm = 2.441555 | Time: 38.0153sec | Token/sec: 1723.939
    Step:    89 | Loss: 6.2451 | lr = 0.000590 | Norm = 2.533587 | Time: 38.0462sec | Token/sec: 1722.536
    Step:    90 | Loss: 6.2008 | lr = 0.000589 | Norm = 4.254522 | Time: 38.0292sec | Token/sec: 1723.308
    Step:    91 | Loss: 6.2007 | lr = 0.000589 | Norm = 2.785007 | Time: 38.0360sec | Token/sec: 1722.999
    Step:    92 | Loss: 6.1739 | lr = 0.000589 | Norm = 2.470515 | Time: 37.9624sec | Token/sec: 1726.341
    Step:    93 | Loss: 6.1537 | lr = 0.000588 | Norm = 2.573724 | Time: 37.9346sec | Token/sec: 1727.605
    Step:    94 | Loss: 6.4335 | lr = 0.000588 | Norm = 4.526144 | Time: 37.8875sec | Token/sec: 1729.754
    Step:    95 | Loss: 6.1558 | lr = 0.000588 | Norm = 2.131199 | Time: 37.9808sec | Token/sec: 1725.503
    Step:    96 | Loss: 6.0725 | lr = 0.000588 | Norm = 2.397848 | Time: 37.9995sec | Token/sec: 1724.653
    Step:    97 | Loss: 6.1288 | lr = 0.000587 | Norm = 3.170344 | Time: 38.0355sec | Token/sec: 1723.024
    Step:    98 | Loss: 6.2032 | lr = 0.000587 | Norm = 3.228719 | Time: 38.0280sec | Token/sec: 1723.363
    Step:    99 | Loss: 6.0750 | lr = 0.000587 | Norm = 2.873083 | Time: 38.0264sec | Token/sec: 1723.433
    Validation loss: 6.1464
    Rank: 1 | Sample1: I am crazy man, the second term into the city's book time when the Indian school; the most popular of his career with the city of war in a city of the north was the man in
    Rank: 1 | Sample2: I am crazy man, I, it comes to say the city and had been been been built.
    Why
    In the right, the fact. He took a new man and the first. The
    Rank: 1 | Sample3: I am crazy man, who, many more information at school school and,’s new teacher teacher students in them all that the students they also learn to the course with the students for all all
    Rank: 0 | Sample1: I am crazy man, there to know in place. It is not have a bad to come about it. When he tried to the old in the person by in the first-old he told and
    Rank: 0 | Sample2: I am crazy man, but in an un-year-BI want to come back, but the world-known the other time, so I wrote about it it’t believe. He
    Rank: 0 | Sample3: I am crazy man, who had been to get to bring to their them to the same time when they had to have come from those to go to make their way it and you’s�
    HellaSwag accuracy: 2383/10042=0.2373
    Step:   100 | Loss: 6.1137 | lr = 0.000586 | Norm = 2.945970 | Time: 358.7247sec | Token/sec: 182.692
    Step:   101 | Loss: 5.9787 | lr = 0.000586 | Norm = 2.644571 | Time: 38.2641sec | Token/sec: 1712.729
    Step:   102 | Loss: 6.0591 | lr = 0.000586 | Norm = 2.723456 | Time: 38.2837sec | Token/sec: 1711.850
    Step:   103 | Loss: 6.0297 | lr = 0.000586 | Norm = 2.529769 | Time: 38.1066sec | Token/sec: 1719.806
    Step:   104 | Loss: 6.1347 | lr = 0.000585 | Norm = 3.386575 | Time: 38.0933sec | Token/sec: 1720.406
    Step:   105 | Loss: 6.0954 | lr = 0.000585 | Norm = 2.487685 | Time: 38.1186sec | Token/sec: 1719.264
    Step:   106 | Loss: 5.9943 | lr = 0.000585 | Norm = 2.794496 | Time: 38.3732sec | Token/sec: 1707.861
    Step:   107 | Loss: 6.0096 | lr = 0.000584 | Norm = 1.839992 | Time: 37.7851sec | Token/sec: 1734.439
    Step:   108 | Loss: 5.9894 | lr = 0.000584 | Norm = 2.731471 | Time: 38.2857sec | Token/sec: 1711.762
    Step:   109 | Loss: 6.0029 | lr = 0.000584 | Norm = 2.398769 | Time: 38.0753sec | Token/sec: 1721.220
    Step:   110 | Loss: 5.9928 | lr = 0.000583 | Norm = 2.008768 | Time: 37.6760sec | Token/sec: 1739.464
    Step:   111 | Loss: 6.0319 | lr = 0.000583 | Norm = 2.067659 | Time: 37.8182sec | Token/sec: 1732.921
    Step:   112 | Loss: 6.0030 | lr = 0.000583 | Norm = 3.299997 | Time: 37.9329sec | Token/sec: 1727.682
    Step:   113 | Loss: 6.0000 | lr = 0.000582 | Norm = 2.551481 | Time: 38.0168sec | Token/sec: 1723.868
    Step:   114 | Loss: 5.9925 | lr = 0.000582 | Norm = 2.256890 | Time: 38.0339sec | Token/sec: 1723.093
    Step:   115 | Loss: 5.9265 | lr = 0.000582 | Norm = 3.799952 | Time: 38.1009sec | Token/sec: 1720.062
    Step:   116 | Loss: 5.8968 | lr = 0.000581 | Norm = 2.359511 | Time: 38.0942sec | Token/sec: 1720.369
    Step:   117 | Loss: 5.9380 | lr = 0.000581 | Norm = 2.204690 | Time: 38.0640sec | Token/sec: 1721.731
    Step:   118 | Loss: 5.9293 | lr = 0.000581 | Norm = 2.888983 | Time: 38.1352sec | Token/sec: 1718.518
    Step:   119 | Loss: 5.8595 | lr = 0.000580 | Norm = 3.370170 | Time: 38.1185sec | Token/sec: 1719.270
    Step:   120 | Loss: 6.0198 | lr = 0.000580 | Norm = 2.823677 | Time: 38.0896sec | Token/sec: 1720.576
    Step:   121 | Loss: 6.0869 | lr = 0.000580 | Norm = 2.892804 | Time: 37.9781sec | Token/sec: 1725.624
    Step:   122 | Loss: 6.0364 | lr = 0.000579 | Norm = 2.093232 | Time: 37.8726sec | Token/sec: 1730.434
    Step:   123 | Loss: 5.9063 | lr = 0.000579 | Norm = 1.761348 | Time: 37.7198sec | Token/sec: 1737.442
    Step:   124 | Loss: 5.9526 | lr = 0.000578 | Norm = 2.802904 | Time: 37.9022sec | Token/sec: 1729.080
    Step:   125 | Loss: 5.9867 | lr = 0.000578 | Norm = 2.563827 | Time: 38.0339sec | Token/sec: 1723.094
    Step:   126 | Loss: 5.9528 | lr = 0.000578 | Norm = 2.529243 | Time: 37.9857sec | Token/sec: 1725.281
    Step:   127 | Loss: 5.9018 | lr = 0.000577 | Norm = 2.493259 | Time: 38.0203sec | Token/sec: 1723.709
    Step:   128 | Loss: 5.8857 | lr = 0.000577 | Norm = 1.814809 | Time: 37.9502sec | Token/sec: 1726.894
    Step:   129 | Loss: 5.9083 | lr = 0.000577 | Norm = 3.267824 | Time: 37.9414sec | Token/sec: 1727.294
    Step:   130 | Loss: 5.9967 | lr = 0.000576 | Norm = 2.344712 | Time: 37.9597sec | Token/sec: 1726.463
    Step:   131 | Loss: 5.9932 | lr = 0.000576 | Norm = 2.779012 | Time: 38.0077sec | Token/sec: 1724.284
    Step:   132 | Loss: 6.0149 | lr = 0.000575 | Norm = 2.640066 | Time: 37.9903sec | Token/sec: 1725.070
    Step:   133 | Loss: 5.9149 | lr = 0.000575 | Norm = 3.068473 | Time: 37.9684sec | Token/sec: 1726.068
    Step:   134 | Loss: 5.9846 | lr = 0.000575 | Norm = 2.625440 | Time: 38.0115sec | Token/sec: 1724.109
    Step:   135 | Loss: 5.8455 | lr = 0.000574 | Norm = 2.827973 | Time: 37.9537sec | Token/sec: 1726.737
    Step:   136 | Loss: 5.9072 | lr = 0.000574 | Norm = 2.573128 | Time: 37.9863sec | Token/sec: 1725.254
    Step:   137 | Loss: 5.7841 | lr = 0.000573 | Norm = 2.196847 | Time: 38.0428sec | Token/sec: 1722.689
    Step:   138 | Loss: 5.8086 | lr = 0.000573 | Norm = 2.142322 | Time: 37.9824sec | Token/sec: 1725.433
    Step:   139 | Loss: 5.8150 | lr = 0.000572 | Norm = 2.737393 | Time: 37.9214sec | Token/sec: 1728.205
    Step:   140 | Loss: 5.7839 | lr = 0.000572 | Norm = 3.380405 | Time: 37.9803sec | Token/sec: 1725.527
    Step:   141 | Loss: 5.7514 | lr = 0.000572 | Norm = 2.092735 | Time: 37.9805sec | Token/sec: 1725.517
    Step:   142 | Loss: 5.8329 | lr = 0.000571 | Norm = 2.190169 | Time: 37.9934sec | Token/sec: 1724.932
    Step:   143 | Loss: 5.8004 | lr = 0.000571 | Norm = 3.429219 | Time: 38.0440sec | Token/sec: 1722.637
    Step:   144 | Loss: 5.7827 | lr = 0.000570 | Norm = 2.812196 | Time: 38.0688sec | Token/sec: 1721.517
    Step:   145 | Loss: 5.7844 | lr = 0.000570 | Norm = 3.006187 | Time: 38.0380sec | Token/sec: 1722.910
    Step:   146 | Loss: 5.9699 | lr = 0.000569 | Norm = 2.712579 | Time: 38.0535sec | Token/sec: 1722.207
    Step:   147 | Loss: 5.8066 | lr = 0.000569 | Norm = 3.110607 | Time: 38.0560sec | Token/sec: 1722.094
    Step:   148 | Loss: 5.7705 | lr = 0.000569 | Norm = 2.515055 | Time: 37.9702sec | Token/sec: 1725.984
    Step:   149 | Loss: 5.6362 | lr = 0.000568 | Norm = 2.562815 | Time: 38.0486sec | Token/sec: 1722.428
    Step:   150 | Loss: 5.8474 | lr = 0.000568 | Norm = 2.872615 | Time: 38.0537sec | Token/sec: 1722.199
    Step:   151 | Loss: 5.6675 | lr = 0.000567 | Norm = 3.370335 | Time: 38.0287sec | Token/sec: 1723.329
    Step:   152 | Loss: 5.7371 | lr = 0.000567 | Norm = 4.454326 | Time: 38.0258sec | Token/sec: 1723.463
    Step:   153 | Loss: 5.6041 | lr = 0.000566 | Norm = 4.503344 | Time: 38.0249sec | Token/sec: 1723.501
    Step:   154 | Loss: 5.7786 | lr = 0.000566 | Norm = 2.792457 | Time: 38.0270sec | Token/sec: 1723.406
    Step:   155 | Loss: 5.8784 | lr = 0.000565 | Norm = 2.461708 | Time: 38.1279sec | Token/sec: 1718.847
    Step:   156 | Loss: 5.7276 | lr = 0.000565 | Norm = 2.081311 | Time: 38.2266sec | Token/sec: 1714.407
    Step:   157 | Loss: 5.8087 | lr = 0.000564 | Norm = 2.322462 | Time: 38.1511sec | Token/sec: 1717.802
    Step:   158 | Loss: 5.7753 | lr = 0.000564 | Norm = 2.606029 | Time: 38.1381sec | Token/sec: 1718.388
    Step:   159 | Loss: 5.7999 | lr = 0.000564 | Norm = 2.756965 | Time: 38.0991sec | Token/sec: 1720.145
    Step:   160 | Loss: 5.7136 | lr = 0.000563 | Norm = 2.401997 | Time: 38.0923sec | Token/sec: 1720.454
    Step:   161 | Loss: 5.6848 | lr = 0.000563 | Norm = 2.377895 | Time: 38.0697sec | Token/sec: 1721.474
    Step:   162 | Loss: 5.8279 | lr = 0.000562 | Norm = 1.893875 | Time: 38.0662sec | Token/sec: 1721.634
    Step:   163 | Loss: 5.7308 | lr = 0.000562 | Norm = 2.394960 | Time: 38.0841sec | Token/sec: 1720.825
    Step:   164 | Loss: 5.6703 | lr = 0.000561 | Norm = 2.788254 | Time: 38.0819sec | Token/sec: 1720.922
    Step:   165 | Loss: 5.7615 | lr = 0.000561 | Norm = 2.939850 | Time: 38.0748sec | Token/sec: 1721.246
    Step:   166 | Loss: 5.7826 | lr = 0.000560 | Norm = 4.988278 | Time: 38.0752sec | Token/sec: 1721.226
    Step:   167 | Loss: 5.8049 | lr = 0.000560 | Norm = 4.030536 | Time: 38.0751sec | Token/sec: 1721.230
    Step:   168 | Loss: 5.6459 | lr = 0.000559 | Norm = 2.938787 | Time: 38.0599sec | Token/sec: 1721.918
    Step:   169 | Loss: 5.6984 | lr = 0.000559 | Norm = 2.538375 | Time: 38.0189sec | Token/sec: 1723.776
    Step:   170 | Loss: 5.6735 | lr = 0.000558 | Norm = 2.258814 | Time: 37.9493sec | Token/sec: 1726.938
    Step:   171 | Loss: 5.6823 | lr = 0.000558 | Norm = 3.265758 | Time: 37.9430sec | Token/sec: 1727.223
    Step:   172 | Loss: 5.7006 | lr = 0.000557 | Norm = 2.539157 | Time: 37.9061sec | Token/sec: 1728.902
    Step:   173 | Loss: 5.6828 | lr = 0.000557 | Norm = 2.960557 | Time: 37.9125sec | Token/sec: 1728.612
    Step:   174 | Loss: 5.8131 | lr = 0.000556 | Norm = 3.173682 | Time: 37.9491sec | Token/sec: 1726.945
    Step:   175 | Loss: 5.6655 | lr = 0.000555 | Norm = 7.869212 | Time: 37.9287sec | Token/sec: 1727.872
    Step:   176 | Loss: 5.7412 | lr = 0.000555 | Norm = 2.301804 | Time: 37.9676sec | Token/sec: 1726.105
    Step:   177 | Loss: 5.6954 | lr = 0.000554 | Norm = 2.982972 | Time: 37.9589sec | Token/sec: 1726.499
    Step:   178 | Loss: 5.7025 | lr = 0.000554 | Norm = 2.690406 | Time: 38.0142sec | Token/sec: 1723.985
    Step:   179 | Loss: 5.6179 | lr = 0.000553 | Norm = 2.521697 | Time: 38.0201sec | Token/sec: 1723.721
    Step:   180 | Loss: 5.6896 | lr = 0.000553 | Norm = 2.368711 | Time: 38.0667sec | Token/sec: 1721.609
    Step:   181 | Loss: 5.6117 | lr = 0.000552 | Norm = 2.617119 | Time: 38.0393sec | Token/sec: 1722.851
    Step:   182 | Loss: 5.7214 | lr = 0.000552 | Norm = 1.810223 | Time: 38.0645sec | Token/sec: 1721.708
    Step:   183 | Loss: 5.6701 | lr = 0.000551 | Norm = 2.377648 | Time: 38.0615sec | Token/sec: 1721.844
    Step:   184 | Loss: 5.5971 | lr = 0.000551 | Norm = 1.982086 | Time: 38.0692sec | Token/sec: 1721.494
    Step:   185 | Loss: 5.6194 | lr = 0.000550 | Norm = 2.208912 | Time: 38.0791sec | Token/sec: 1721.048
    Step:   186 | Loss: 5.7387 | lr = 0.000550 | Norm = 2.399647 | Time: 38.1415sec | Token/sec: 1718.233
    Step:   187 | Loss: 5.8718 | lr = 0.000549 | Norm = 3.262118 | Time: 38.2415sec | Token/sec: 1713.738
    Step:   188 | Loss: 5.7605 | lr = 0.000548 | Norm = 2.426478 | Time: 38.2031sec | Token/sec: 1715.464
    Step:   189 | Loss: 5.8625 | lr = 0.000548 | Norm = 2.371684 | Time: 38.0064sec | Token/sec: 1724.339
    Step:   190 | Loss: 5.8386 | lr = 0.000547 | Norm = 2.597431 | Time: 37.9205sec | Token/sec: 1728.249
    Step:   191 | Loss: 5.8141 | lr = 0.000547 | Norm = 3.362817 | Time: 37.8346sec | Token/sec: 1732.170
    Step:   192 | Loss: 5.8843 | lr = 0.000546 | Norm = 2.119435 | Time: 37.8981sec | Token/sec: 1729.270
    Step:   193 | Loss: 5.7586 | lr = 0.000546 | Norm = 2.989460 | Time: 37.9012sec | Token/sec: 1729.128
    Step:   194 | Loss: 5.8685 | lr = 0.000545 | Norm = 3.051245 | Time: 37.8660sec | Token/sec: 1730.735
    Step:   195 | Loss: 5.8464 | lr = 0.000544 | Norm = 3.128039 | Time: 37.8654sec | Token/sec: 1730.762
    Step:   196 | Loss: 5.7492 | lr = 0.000544 | Norm = 2.434625 | Time: 37.9646sec | Token/sec: 1726.239
    Step:   197 | Loss: 5.7642 | lr = 0.000543 | Norm = 2.007415 | Time: 37.9880sec | Token/sec: 1725.178
    Step:   198 | Loss: 5.7316 | lr = 0.000543 | Norm = 2.293479 | Time: 38.0343sec | Token/sec: 1723.076
    Step:   199 | Loss: 5.8446 | lr = 0.000542 | Norm = 2.929621 | Time: 38.0535sec | Token/sec: 1722.207
    Validation loss: 5.7390
    Rank: 1 | Sample1: I am crazy man, it to go
    The last year from our bodies of Jesus has been, is the world is important in this is actually very few, with a small, and which can't
    Rank: 1 | Sample2: I am crazy man, about it was no doubt that there not have been doing in. She was much more but we think I” was not just as the man who to say that some God
    Rank: 1 | Sample3: I am crazy man, and he came from his return to his he he he he he moved; so he he he him his great his father he he shall he he he he was his in he
    Rank: 0 | Sample1: I am crazy man, I am. I am going to put me that I, my my I still I am I am my my my my I my I have I I I I'm I am
    Rank: 0 | Sample2: I am crazy man, a whole of the world on the power a series of man and the next century.Sawattle of the word (and a man – was in both with a man
    Rank: 0 | Sample3: I am crazy man, if not say about about what’s life. We have to believe I think about that you say and the only. But I think I knew something I think about me
    HellaSwag accuracy: 2427/10042=0.2417
    Step:   200 | Loss: 5.8364 | lr = 0.000542 | Norm = 2.656257 | Time: 359.0439sec | Token/sec: 182.529
    Step:   201 | Loss: 5.7368 | lr = 0.000541 | Norm = 2.418694 | Time: 38.2553sec | Token/sec: 1713.124
    Step:   202 | Loss: 5.8645 | lr = 0.000540 | Norm = 2.385367 | Time: 38.3151sec | Token/sec: 1710.448
    Step:   203 | Loss: 5.7650 | lr = 0.000540 | Norm = 2.437188 | Time: 38.0579sec | Token/sec: 1722.008
    Step:   204 | Loss: 5.7491 | lr = 0.000539 | Norm = 2.160355 | Time: 37.7916sec | Token/sec: 1734.144
    Step:   205 | Loss: 5.7677 | lr = 0.000539 | Norm = 2.382365 | Time: 37.7396sec | Token/sec: 1736.532
    Step:   206 | Loss: 5.8147 | lr = 0.000538 | Norm = 2.091051 | Time: 37.6887sec | Token/sec: 1738.878
    Step:   207 | Loss: 5.7363 | lr = 0.000537 | Norm = 2.850652 | Time: 37.7468sec | Token/sec: 1736.198
    Step:   208 | Loss: 5.7596 | lr = 0.000537 | Norm = 2.805882 | Time: 37.9338sec | Token/sec: 1727.643
    Step:   209 | Loss: 5.8387 | lr = 0.000536 | Norm = 2.334733 | Time: 37.8759sec | Token/sec: 1730.280
    Step:   210 | Loss: 5.7872 | lr = 0.000535 | Norm = 2.126207 | Time: 37.8640sec | Token/sec: 1730.824
    Step:   211 | Loss: 5.7175 | lr = 0.000535 | Norm = 2.134197 | Time: 38.0272sec | Token/sec: 1723.399
    Step:   212 | Loss: 5.7510 | lr = 0.000534 | Norm = 2.604255 | Time: 38.0717sec | Token/sec: 1721.384
    Step:   213 | Loss: 5.6918 | lr = 0.000534 | Norm = 2.270287 | Time: 38.0867sec | Token/sec: 1720.706
    Step:   214 | Loss: 5.7236 | lr = 0.000533 | Norm = 2.138803 | Time: 38.0979sec | Token/sec: 1720.202
    Step:   215 | Loss: 5.6361 | lr = 0.000532 | Norm = 2.285562 | Time: 38.0332sec | Token/sec: 1723.126
    Step:   216 | Loss: 5.6385 | lr = 0.000532 | Norm = 2.478571 | Time: 37.9706sec | Token/sec: 1725.965
    Step:   217 | Loss: 5.7109 | lr = 0.000531 | Norm = 2.531244 | Time: 37.9769sec | Token/sec: 1725.682
    Step:   218 | Loss: 5.5998 | lr = 0.000530 | Norm = 2.109442 | Time: 37.9621sec | Token/sec: 1726.354
    Step:   219 | Loss: 5.8376 | lr = 0.000530 | Norm = 2.345816 | Time: 37.9361sec | Token/sec: 1727.538
    Step:   220 | Loss: 5.7605 | lr = 0.000529 | Norm = 2.421527 | Time: 37.9412sec | Token/sec: 1727.303
    Step:   221 | Loss: 5.6868 | lr = 0.000529 | Norm = 3.969023 | Time: 37.8911sec | Token/sec: 1729.587
    Step:   222 | Loss: 5.7033 | lr = 0.000528 | Norm = 2.301201 | Time: 37.8713sec | Token/sec: 1730.493
    Step:   223 | Loss: 5.6984 | lr = 0.000527 | Norm = 2.157001 | Time: 37.9184sec | Token/sec: 1728.344
    Step:   224 | Loss: 5.6277 | lr = 0.000527 | Norm = 2.358566 | Time: 37.8600sec | Token/sec: 1731.007
    Step:   225 | Loss: 5.7190 | lr = 0.000526 | Norm = 2.434090 | Time: 37.8596sec | Token/sec: 1731.026
    Step:   226 | Loss: 5.7396 | lr = 0.000525 | Norm = 2.802672 | Time: 37.8750sec | Token/sec: 1730.325
    Step:   227 | Loss: 5.7197 | lr = 0.000525 | Norm = 2.193663 | Time: 37.8764sec | Token/sec: 1730.262
    Step:   228 | Loss: 5.7158 | lr = 0.000524 | Norm = 1.925692 | Time: 37.9003sec | Token/sec: 1729.171
    Step:   229 | Loss: 5.7150 | lr = 0.000523 | Norm = 2.239949 | Time: 37.9078sec | Token/sec: 1728.825
    Step:   230 | Loss: 5.6959 | lr = 0.000523 | Norm = 1.820702 | Time: 37.8901sec | Token/sec: 1729.633
    Step:   231 | Loss: 5.6682 | lr = 0.000522 | Norm = 2.276253 | Time: 37.8898sec | Token/sec: 1729.647
    Step:   232 | Loss: 5.6038 | lr = 0.000521 | Norm = 2.261981 | Time: 37.8715sec | Token/sec: 1730.483
    Step:   233 | Loss: 5.6933 | lr = 0.000521 | Norm = 2.445355 | Time: 37.8697sec | Token/sec: 1730.564
    Step:   234 | Loss: 5.5838 | lr = 0.000520 | Norm = 2.639182 | Time: 37.8827sec | Token/sec: 1729.971
    Step:   235 | Loss: 5.6391 | lr = 0.000519 | Norm = 2.956489 | Time: 37.8589sec | Token/sec: 1731.062
    Step:   236 | Loss: 5.6065 | lr = 0.000519 | Norm = 2.642512 | Time: 37.8379sec | Token/sec: 1732.018
    Step:   237 | Loss: 5.6532 | lr = 0.000518 | Norm = 2.649867 | Time: 37.8738sec | Token/sec: 1730.379
    Step:   238 | Loss: 5.6213 | lr = 0.000517 | Norm = 2.578210 | Time: 37.8577sec | Token/sec: 1731.112
    Step:   239 | Loss: 5.6246 | lr = 0.000516 | Norm = 3.072216 | Time: 37.7654sec | Token/sec: 1735.347
    Step:   240 | Loss: 5.6752 | lr = 0.000516 | Norm = 2.875970 | Time: 37.8101sec | Token/sec: 1733.296
    Step:   241 | Loss: 5.6848 | lr = 0.000515 | Norm = 2.292024 | Time: 37.8970sec | Token/sec: 1729.320
    Step:   242 | Loss: 5.6237 | lr = 0.000514 | Norm = 2.866470 | Time: 37.8858sec | Token/sec: 1729.828
    Step:   243 | Loss: 5.5880 | lr = 0.000514 | Norm = 2.400892 | Time: 37.9367sec | Token/sec: 1727.511
    Step:   244 | Loss: 5.6266 | lr = 0.000513 | Norm = 2.315367 | Time: 37.9137sec | Token/sec: 1728.558
    Step:   245 | Loss: 5.4273 | lr = 0.000512 | Norm = 2.648910 | Time: 37.9441sec | Token/sec: 1727.173
    Step:   246 | Loss: 5.7170 | lr = 0.000512 | Norm = 3.220392 | Time: 37.9074sec | Token/sec: 1728.847
    Step:   247 | Loss: 5.6255 | lr = 0.000511 | Norm = 2.530946 | Time: 37.9094sec | Token/sec: 1728.755
    Step:   248 | Loss: 5.5723 | lr = 0.000510 | Norm = 2.046163 | Time: 37.9802sec | Token/sec: 1725.531
    Step:   249 | Loss: 5.5627 | lr = 0.000509 | Norm = 2.025889 | Time: 38.2236sec | Token/sec: 1714.545
    Step:   250 | Loss: 5.6441 | lr = 0.000509 | Norm = 2.065288 | Time: 38.3535sec | Token/sec: 1708.738
    Step:   251 | Loss: 5.6107 | lr = 0.000508 | Norm = 2.651454 | Time: 38.1181sec | Token/sec: 1719.288
    Step:   252 | Loss: 5.6900 | lr = 0.000507 | Norm = 2.611567 | Time: 37.9093sec | Token/sec: 1728.760
    Step:   253 | Loss: 5.6017 | lr = 0.000507 | Norm = 2.324898 | Time: 37.8104sec | Token/sec: 1733.278
    Step:   254 | Loss: 5.5364 | lr = 0.000506 | Norm = 2.226419 | Time: 37.7913sec | Token/sec: 1734.154
    Step:   255 | Loss: 5.5246 | lr = 0.000505 | Norm = 2.359229 | Time: 37.9425sec | Token/sec: 1727.247
    Step:   256 | Loss: 5.6256 | lr = 0.000504 | Norm = 2.202682 | Time: 37.9799sec | Token/sec: 1725.546
    Step:   257 | Loss: 5.5744 | lr = 0.000504 | Norm = 2.093377 | Time: 37.9433sec | Token/sec: 1727.209
    Step:   258 | Loss: 5.5265 | lr = 0.000503 | Norm = 2.147517 | Time: 37.9532sec | Token/sec: 1726.757
    Step:   259 | Loss: 5.6086 | lr = 0.000502 | Norm = 2.041713 | Time: 37.9250sec | Token/sec: 1728.042
    Step:   260 | Loss: 5.5940 | lr = 0.000502 | Norm = 2.910277 | Time: 37.8934sec | Token/sec: 1729.484
    Step:   261 | Loss: 5.6346 | lr = 0.000501 | Norm = 2.374249 | Time: 37.9654sec | Token/sec: 1726.203
    Step:   262 | Loss: 5.6975 | lr = 0.000500 | Norm = 2.816488 | Time: 37.9809sec | Token/sec: 1725.498
    Step:   263 | Loss: 5.6363 | lr = 0.000499 | Norm = 3.028950 | Time: 38.0095sec | Token/sec: 1724.201
    Step:   264 | Loss: 5.5759 | lr = 0.000499 | Norm = 3.151110 | Time: 38.0023sec | Token/sec: 1724.525
    Step:   265 | Loss: 5.5807 | lr = 0.000498 | Norm = 2.373384 | Time: 38.0136sec | Token/sec: 1724.013
    Step:   266 | Loss: 5.5269 | lr = 0.000497 | Norm = 2.026010 | Time: 38.1261sec | Token/sec: 1718.930
    Step:   267 | Loss: 5.5820 | lr = 0.000496 | Norm = 2.732610 | Time: 38.3029sec | Token/sec: 1710.994
    Step:   268 | Loss: 5.6500 | lr = 0.000496 | Norm = 3.132042 | Time: 38.1004sec | Token/sec: 1720.089
    Step:   269 | Loss: 5.6120 | lr = 0.000495 | Norm = 2.377822 | Time: 37.7142sec | Token/sec: 1737.701
    Step:   270 | Loss: 5.6884 | lr = 0.000494 | Norm = 2.523684 | Time: 37.9978sec | Token/sec: 1724.730
    Step:   271 | Loss: 5.5932 | lr = 0.000493 | Norm = 3.784343 | Time: 38.2268sec | Token/sec: 1714.400
    Step:   272 | Loss: 5.5863 | lr = 0.000493 | Norm = 2.728063 | Time: 38.1394sec | Token/sec: 1718.327
    Step:   273 | Loss: 5.5980 | lr = 0.000492 | Norm = 2.874999 | Time: 37.9035sec | Token/sec: 1729.021
    Step:   274 | Loss: 5.6101 | lr = 0.000491 | Norm = 2.704723 | Time: 37.7880sec | Token/sec: 1734.306
    Step:   275 | Loss: 5.5911 | lr = 0.000490 | Norm = 2.466421 | Time: 37.7609sec | Token/sec: 1735.553
    Step:   276 | Loss: 5.5718 | lr = 0.000489 | Norm = 2.002949 | Time: 37.8796sec | Token/sec: 1730.113
    Step:   277 | Loss: 5.4076 | lr = 0.000489 | Norm = 2.503273 | Time: 37.8857sec | Token/sec: 1729.836
    Step:   278 | Loss: 5.5493 | lr = 0.000488 | Norm = 2.965683 | Time: 37.9526sec | Token/sec: 1726.784
    Step:   279 | Loss: 5.4550 | lr = 0.000487 | Norm = 2.295872 | Time: 37.9335sec | Token/sec: 1727.653
    Step:   280 | Loss: 5.5732 | lr = 0.000486 | Norm = 3.004120 | Time: 37.9429sec | Token/sec: 1727.227
    Step:   281 | Loss: 5.5189 | lr = 0.000486 | Norm = 2.529293 | Time: 37.9647sec | Token/sec: 1726.234
    Step:   282 | Loss: 5.4855 | lr = 0.000485 | Norm = 2.024548 | Time: 38.0070sec | Token/sec: 1724.314
    Step:   283 | Loss: 5.4441 | lr = 0.000484 | Norm = 2.384448 | Time: 37.9900sec | Token/sec: 1725.085
    Step:   284 | Loss: 5.5170 | lr = 0.000483 | Norm = 2.314052 | Time: 38.0248sec | Token/sec: 1723.508
    Step:   285 | Loss: 5.5329 | lr = 0.000482 | Norm = 2.962928 | Time: 38.0454sec | Token/sec: 1722.574
    Step:   286 | Loss: 5.5025 | lr = 0.000482 | Norm = 2.661259 | Time: 38.0742sec | Token/sec: 1721.270
    Step:   287 | Loss: 5.5401 | lr = 0.000481 | Norm = 2.274850 | Time: 38.3446sec | Token/sec: 1709.131
    Step:   288 | Loss: 5.4607 | lr = 0.000480 | Norm = 2.781744 | Time: 37.9199sec | Token/sec: 1728.277
    Step:   289 | Loss: 5.5063 | lr = 0.000479 | Norm = 2.647386 | Time: 38.0382sec | Token/sec: 1722.899
    Step:   290 | Loss: 5.5330 | lr = 0.000479 | Norm = 3.060319 | Time: 38.2931sec | Token/sec: 1711.430
    Step:   291 | Loss: 5.4473 | lr = 0.000478 | Norm = 2.715728 | Time: 38.1826sec | Token/sec: 1716.385
    Step:   292 | Loss: 5.4489 | lr = 0.000477 | Norm = 2.434325 | Time: 38.2058sec | Token/sec: 1715.342
    Step:   293 | Loss: 5.5209 | lr = 0.000476 | Norm = 2.738749 | Time: 38.2132sec | Token/sec: 1715.011
    Step:   294 | Loss: 5.5186 | lr = 0.000475 | Norm = 2.708067 | Time: 38.1810sec | Token/sec: 1716.454
    Step:   295 | Loss: 5.5234 | lr = 0.000475 | Norm = 3.741621 | Time: 38.1891sec | Token/sec: 1716.090
    Step:   296 | Loss: 5.4536 | lr = 0.000474 | Norm = 3.112809 | Time: 38.1704sec | Token/sec: 1716.931
    Step:   297 | Loss: 5.4186 | lr = 0.000473 | Norm = 2.462995 | Time: 38.2074sec | Token/sec: 1715.269
    Step:   298 | Loss: 5.4512 | lr = 0.000472 | Norm = 2.437735 | Time: 38.1758sec | Token/sec: 1716.688
    Step:   299 | Loss: 5.4513 | lr = 0.000471 | Norm = 2.331041 | Time: 38.2074sec | Token/sec: 1715.269
    Validation loss: 5.5239
    Rank: 1 | Sample1: I am crazy man, in his heart of what was born. The he became a woman was married married who was born in 15. He told I had been married.
    I had been in the
    Rank: 1 | Sample2: I am crazy man, or all the great opportunity of what you. But you may take care of life because you want a smile can have a big picture
    My favorite people, people at your house
    Rank: 1 | Sample3: I am crazy man, a little room so very old one to me as for many more in your child will do have one of your child or to feel them, when you're not like your child
    Rank: 0 | Sample1: I am crazy man, you do so you’re still a little ones, or too. Your baby was also the man’s. She asked that you’s not, the
    Rank: 0 | Sample2: I am crazy man, you’t mean?’s one of your heart, you”
    You might not get the case of your question!
    If you’re looking
    Rank: 0 | Sample3: I am crazy man, he was only. He then he used for him for his, it with a man's in his son, his brothers.
    The man’s and his daughter of
    HellaSwag accuracy: 2439/10042=0.2429
    Step:   300 | Loss: 5.4927 | lr = 0.000470 | Norm = 2.700323 | Time: 357.9534sec | Token/sec: 183.085
    Step:   301 | Loss: 5.5652 | lr = 0.000470 | Norm = 2.741217 | Time: 37.7919sec | Token/sec: 1734.128
    Step:   302 | Loss: 5.4262 | lr = 0.000469 | Norm = 2.982422 | Time: 37.8101sec | Token/sec: 1733.295
    Step:   303 | Loss: 5.4723 | lr = 0.000468 | Norm = 2.128600 | Time: 37.7524sec | Token/sec: 1735.942
    Step:   304 | Loss: 5.3972 | lr = 0.000467 | Norm = 2.656728 | Time: 37.9235sec | Token/sec: 1728.112
    Step:   305 | Loss: 5.4248 | lr = 0.000466 | Norm = 2.704168 | Time: 38.1366sec | Token/sec: 1718.455
    Step:   306 | Loss: 5.4520 | lr = 0.000466 | Norm = 2.857405 | Time: 38.3218sec | Token/sec: 1710.148
    Step:   307 | Loss: 5.5006 | lr = 0.000465 | Norm = 2.966444 | Time: 38.1714sec | Token/sec: 1716.889
    Step:   308 | Loss: 5.4113 | lr = 0.000464 | Norm = 2.457796 | Time: 37.9941sec | Token/sec: 1724.899
    Step:   309 | Loss: 5.4474 | lr = 0.000463 | Norm = 2.221795 | Time: 37.8292sec | Token/sec: 1732.419
    Step:   310 | Loss: 5.4785 | lr = 0.000462 | Norm = 2.653030 | Time: 37.7587sec | Token/sec: 1735.655
    Step:   311 | Loss: 5.5398 | lr = 0.000461 | Norm = 2.620950 | Time: 37.7379sec | Token/sec: 1736.610
    Step:   312 | Loss: 5.5339 | lr = 0.000461 | Norm = 2.524597 | Time: 37.7402sec | Token/sec: 1736.503
    Step:   313 | Loss: 5.4438 | lr = 0.000460 | Norm = 2.410760 | Time: 37.7751sec | Token/sec: 1734.898
    Step:   314 | Loss: 5.4219 | lr = 0.000459 | Norm = 2.222867 | Time: 37.7808sec | Token/sec: 1734.639
    Step:   315 | Loss: 5.3943 | lr = 0.000458 | Norm = 2.488841 | Time: 37.7961sec | Token/sec: 1733.936
    Step:   316 | Loss: 5.4372 | lr = 0.000457 | Norm = 2.399517 | Time: 37.8359sec | Token/sec: 1732.112
    Step:   317 | Loss: 5.5176 | lr = 0.000456 | Norm = 2.545910 | Time: 37.8745sec | Token/sec: 1730.346
    Step:   318 | Loss: 5.3753 | lr = 0.000456 | Norm = 2.129615 | Time: 37.9282sec | Token/sec: 1727.896
    Step:   319 | Loss: 5.3479 | lr = 0.000455 | Norm = 2.123096 | Time: 37.9463sec | Token/sec: 1727.072
    Step:   320 | Loss: 5.4184 | lr = 0.000454 | Norm = 2.803849 | Time: 37.9742sec | Token/sec: 1725.802
    Step:   321 | Loss: 5.4685 | lr = 0.000453 | Norm = 2.198196 | Time: 37.9681sec | Token/sec: 1726.081
    Step:   322 | Loss: 5.3785 | lr = 0.000452 | Norm = 2.538781 | Time: 37.9750sec | Token/sec: 1725.765
    Step:   323 | Loss: 5.4455 | lr = 0.000451 | Norm = 2.601876 | Time: 38.0411sec | Token/sec: 1722.766
    Step:   324 | Loss: 5.3322 | lr = 0.000450 | Norm = 2.666169 | Time: 37.9894sec | Token/sec: 1725.113
    Step:   325 | Loss: 5.3508 | lr = 0.000450 | Norm = 2.725057 | Time: 38.0033sec | Token/sec: 1724.481
    Step:   326 | Loss: 5.4235 | lr = 0.000449 | Norm = 2.992691 | Time: 38.1711sec | Token/sec: 1716.902
    Step:   327 | Loss: 5.4598 | lr = 0.000448 | Norm = 2.690536 | Time: 38.2560sec | Token/sec: 1713.093
    Step:   328 | Loss: 5.4456 | lr = 0.000447 | Norm = 3.053750 | Time: 38.2861sec | Token/sec: 1711.743
    Step:   329 | Loss: 5.3627 | lr = 0.000446 | Norm = 2.700611 | Time: 38.1161sec | Token/sec: 1719.379
    Step:   330 | Loss: 5.3768 | lr = 0.000445 | Norm = 2.489138 | Time: 37.9995sec | Token/sec: 1724.655
    Step:   331 | Loss: 5.3841 | lr = 0.000444 | Norm = 2.841974 | Time: 37.9623sec | Token/sec: 1726.344
    Step:   332 | Loss: 5.2879 | lr = 0.000444 | Norm = 2.914496 | Time: 37.9207sec | Token/sec: 1728.238
    Step:   333 | Loss: 5.3875 | lr = 0.000443 | Norm = 2.813645 | Time: 37.8706sec | Token/sec: 1730.524
    Step:   334 | Loss: 5.3082 | lr = 0.000442 | Norm = 2.661848 | Time: 37.8295sec | Token/sec: 1732.405
    Step:   335 | Loss: 5.2770 | lr = 0.000441 | Norm = 2.381699 | Time: 37.8073sec | Token/sec: 1733.420
    Step:   336 | Loss: 5.3591 | lr = 0.000440 | Norm = 2.800856 | Time: 37.7670sec | Token/sec: 1735.273
    Step:   337 | Loss: 5.3184 | lr = 0.000439 | Norm = 2.457416 | Time: 37.7425sec | Token/sec: 1736.400
    Step:   338 | Loss: 5.3220 | lr = 0.000438 | Norm = 2.561919 | Time: 37.9099sec | Token/sec: 1728.732
    Step:   339 | Loss: 5.3925 | lr = 0.000437 | Norm = 2.422757 | Time: 37.8756sec | Token/sec: 1730.297
    Step:   340 | Loss: 5.3522 | lr = 0.000437 | Norm = 2.492961 | Time: 37.9619sec | Token/sec: 1726.361
    Step:   341 | Loss: 5.3019 | lr = 0.000436 | Norm = 2.494236 | Time: 38.0117sec | Token/sec: 1724.100
    Step:   342 | Loss: 5.2425 | lr = 0.000435 | Norm = 2.560358 | Time: 38.0159sec | Token/sec: 1723.909
    Step:   343 | Loss: 5.3432 | lr = 0.000434 | Norm = 2.578918 | Time: 37.9578sec | Token/sec: 1726.547
    Step:   344 | Loss: 5.2993 | lr = 0.000433 | Norm = 2.730896 | Time: 37.9701sec | Token/sec: 1725.992
    Step:   345 | Loss: 5.2678 | lr = 0.000432 | Norm = 3.079362 | Time: 37.9927sec | Token/sec: 1724.965
    Step:   346 | Loss: 5.3411 | lr = 0.000431 | Norm = 2.847543 | Time: 38.0132sec | Token/sec: 1724.033
    Step:   347 | Loss: 5.3386 | lr = 0.000430 | Norm = 2.506464 | Time: 37.9986sec | Token/sec: 1724.695
    Step:   348 | Loss: 5.3478 | lr = 0.000430 | Norm = 2.618487 | Time: 38.0018sec | Token/sec: 1724.550
    Step:   349 | Loss: 5.3756 | lr = 0.000429 | Norm = 2.298658 | Time: 38.0059sec | Token/sec: 1724.363
    Step:   350 | Loss: 5.3585 | lr = 0.000428 | Norm = 2.710662 | Time: 38.0028sec | Token/sec: 1724.505
    Step:   351 | Loss: 5.3916 | lr = 0.000427 | Norm = 3.077020 | Time: 37.9835sec | Token/sec: 1725.382
    Step:   352 | Loss: 5.3374 | lr = 0.000426 | Norm = 2.613187 | Time: 38.0128sec | Token/sec: 1724.052
    Step:   353 | Loss: 5.3400 | lr = 0.000425 | Norm = 2.986063 | Time: 38.0732sec | Token/sec: 1721.318
    Step:   354 | Loss: 5.3394 | lr = 0.000424 | Norm = 2.703895 | Time: 38.0406sec | Token/sec: 1722.789
    Step:   355 | Loss: 5.3304 | lr = 0.000423 | Norm = 3.077701 | Time: 38.0500sec | Token/sec: 1722.366
    Step:   356 | Loss: 5.3205 | lr = 0.000422 | Norm = 2.919874 | Time: 38.0259sec | Token/sec: 1723.458
    Step:   357 | Loss: 5.3520 | lr = 0.000422 | Norm = 2.870218 | Time: 38.0631sec | Token/sec: 1721.774
    Step:   358 | Loss: 5.3059 | lr = 0.000421 | Norm = 2.939830 | Time: 38.0289sec | Token/sec: 1723.321
    Step:   359 | Loss: 5.4188 | lr = 0.000420 | Norm = 3.917737 | Time: 38.0794sec | Token/sec: 1721.038
    Step:   360 | Loss: 5.3125 | lr = 0.000419 | Norm = 3.076594 | Time: 38.0408sec | Token/sec: 1722.780
    Step:   361 | Loss: 5.3226 | lr = 0.000418 | Norm = 3.046622 | Time: 38.0125sec | Token/sec: 1724.066
    Step:   362 | Loss: 5.2955 | lr = 0.000417 | Norm = 3.545268 | Time: 38.0778sec | Token/sec: 1721.107
    Step:   363 | Loss: 5.3255 | lr = 0.000416 | Norm = 3.928947 | Time: 38.1593sec | Token/sec: 1717.432
    Step:   364 | Loss: 5.3002 | lr = 0.000415 | Norm = 3.617782 | Time: 38.2062sec | Token/sec: 1715.324
    Step:   365 | Loss: 5.2968 | lr = 0.000414 | Norm = 2.911455 | Time: 38.1090sec | Token/sec: 1719.699
    Step:   366 | Loss: 5.3995 | lr = 0.000413 | Norm = 2.922852 | Time: 38.0355sec | Token/sec: 1723.022
    Step:   367 | Loss: 5.2799 | lr = 0.000413 | Norm = 2.592570 | Time: 37.9199sec | Token/sec: 1728.274
    Step:   368 | Loss: 5.3746 | lr = 0.000412 | Norm = 3.101197 | Time: 37.8676sec | Token/sec: 1730.660
    Step:   369 | Loss: 5.2375 | lr = 0.000411 | Norm = 2.782450 | Time: 37.8143sec | Token/sec: 1733.101
    Step:   370 | Loss: 5.4347 | lr = 0.000410 | Norm = 3.496891 | Time: 37.8326sec | Token/sec: 1732.263
    Step:   371 | Loss: 5.4268 | lr = 0.000409 | Norm = 2.495870 | Time: 37.9167sec | Token/sec: 1728.419
    Step:   372 | Loss: 5.4563 | lr = 0.000408 | Norm = 3.214407 | Time: 37.9725sec | Token/sec: 1725.881
    Step:   373 | Loss: 5.3402 | lr = 0.000407 | Norm = 3.572356 | Time: 38.2924sec | Token/sec: 1711.460
    Step:   374 | Loss: 5.4370 | lr = 0.000406 | Norm = 2.953940 | Time: 37.8564sec | Token/sec: 1731.172
    Step:   375 | Loss: 5.5294 | lr = 0.000405 | Norm = 2.427844 | Time: 37.9883sec | Token/sec: 1725.164
    Step:   376 | Loss: 5.5523 | lr = 0.000404 | Norm = 3.646307 | Time: 38.2197sec | Token/sec: 1714.717
    Step:   377 | Loss: 5.4472 | lr = 0.000403 | Norm = 3.451836 | Time: 38.2987sec | Token/sec: 1711.179
    Step:   378 | Loss: 5.4088 | lr = 0.000402 | Norm = 2.974023 | Time: 38.0558sec | Token/sec: 1722.101
    Step:   379 | Loss: 5.5239 | lr = 0.000402 | Norm = 3.194142 | Time: 37.6772sec | Token/sec: 1739.407
    Step:   380 | Loss: 5.5243 | lr = 0.000401 | Norm = 2.847024 | Time: 37.9873sec | Token/sec: 1725.208
    Step:   381 | Loss: 5.4585 | lr = 0.000400 | Norm = 2.813936 | Time: 38.1771sec | Token/sec: 1716.630
    Step:   382 | Loss: 5.4396 | lr = 0.000399 | Norm = 2.491363 | Time: 38.1907sec | Token/sec: 1716.020
    Step:   383 | Loss: 5.4590 | lr = 0.000398 | Norm = 3.149332 | Time: 38.1690sec | Token/sec: 1716.995
    Step:   384 | Loss: 5.4052 | lr = 0.000397 | Norm = 3.132564 | Time: 38.0115sec | Token/sec: 1724.109
    Step:   385 | Loss: 5.4186 | lr = 0.000396 | Norm = 2.678042 | Time: 37.8758sec | Token/sec: 1730.286
    Step:   386 | Loss: 5.5060 | lr = 0.000395 | Norm = 2.757469 | Time: 37.7871sec | Token/sec: 1734.349
    Step:   387 | Loss: 5.4597 | lr = 0.000394 | Norm = 2.948274 | Time: 37.7108sec | Token/sec: 1737.856
    Step:   388 | Loss: 5.3265 | lr = 0.000393 | Norm = 3.333540 | Time: 37.7546sec | Token/sec: 1735.842
    Step:   389 | Loss: 5.4059 | lr = 0.000392 | Norm = 3.001878 | Time: 38.2738sec | Token/sec: 1712.293
    Step:   390 | Loss: 5.4498 | lr = 0.000391 | Norm = 3.012159 | Time: 38.1258sec | Token/sec: 1718.941
    Step:   391 | Loss: 5.3829 | lr = 0.000390 | Norm = 2.792772 | Time: 37.9670sec | Token/sec: 1726.133
    Step:   392 | Loss: 5.4864 | lr = 0.000390 | Norm = 5.097269 | Time: 37.9447sec | Token/sec: 1727.143
    Step:   393 | Loss: 5.4301 | lr = 0.000389 | Norm = 3.916920 | Time: 37.9427sec | Token/sec: 1727.235
    Step:   394 | Loss: 5.3687 | lr = 0.000388 | Norm = 3.457831 | Time: 37.9400sec | Token/sec: 1727.360
    Step:   395 | Loss: 5.3974 | lr = 0.000387 | Norm = 3.546503 | Time: 37.8834sec | Token/sec: 1729.938
    Step:   396 | Loss: 5.3514 | lr = 0.000386 | Norm = 4.824701 | Time: 37.9011sec | Token/sec: 1729.130
    Step:   397 | Loss: 5.4674 | lr = 0.000385 | Norm = 3.886400 | Time: 37.8630sec | Token/sec: 1730.874
    Step:   398 | Loss: 5.4403 | lr = 0.000384 | Norm = 2.866106 | Time: 37.8749sec | Token/sec: 1730.326
    Step:   399 | Loss: 5.4493 | lr = 0.000383 | Norm = 3.276063 | Time: 37.8663sec | Token/sec: 1730.720
    Validation loss: 5.3706
    Rank: 1 | Sample1: I am crazy man, I know you tell with no more than we do that we need: what do I am? I am I could not have I am at that I do I am my work
    Rank: 1 | Sample2: I am crazy man, or was always like it; if I'm it got much like I like my heart and me my my I was doing to go. I felt I think I haven.

    Rank: 1 | Sample3: I am crazy man, is about that our minds are being able to give us and my mind." I am going on my my own my mother, I'm ready at my my next post my mom
    Rank: 0 | Sample1: I am crazy man, he had been brought in him.
    He had to do so do."
    So I know, and his life
    We asked him the heart of this
    As of us
    Rank: 0 | Sample2: I am crazy man, to take place you have, the place in a large, as well-up of an unconcern, and so we can really that I know that I have my own
    Rank: 0 | Sample3: I am crazy man, she decided to have a large man so large scale it can even the big thing you have made you on something that there for you wouldn’t. Let’t
    HellaSwag accuracy: 2460/10042=0.2450
    Step:   400 | Loss: 5.4814 | lr = 0.000382 | Norm = 3.817094 | Time: 356.5478sec | Token/sec: 183.807
    Step:   401 | Loss: 5.3662 | lr = 0.000381 | Norm = 3.342898 | Time: 38.2495sec | Token/sec: 1713.382
    Step:   402 | Loss: 5.3944 | lr = 0.000380 | Norm = 2.732774 | Time: 38.1815sec | Token/sec: 1716.435
    Step:   403 | Loss: 5.4729 | lr = 0.000379 | Norm = 2.957653 | Time: 38.1219sec | Token/sec: 1719.116
    Step:   404 | Loss: 5.3558 | lr = 0.000378 | Norm = 2.787809 | Time: 38.0878sec | Token/sec: 1720.658
    Step:   405 | Loss: 5.4152 | lr = 0.000377 | Norm = 2.857466 | Time: 38.0340sec | Token/sec: 1723.091
    Step:   406 | Loss: 5.4003 | lr = 0.000376 | Norm = 2.829624 | Time: 38.0009sec | Token/sec: 1724.592
    Step:   407 | Loss: 5.3975 | lr = 0.000376 | Norm = 2.458455 | Time: 37.9287sec | Token/sec: 1727.871
    Step:   408 | Loss: 5.3745 | lr = 0.000375 | Norm = 3.046741 | Time: 37.8478sec | Token/sec: 1731.565
    Step:   409 | Loss: 5.3459 | lr = 0.000374 | Norm = 2.753741 | Time: 37.9237sec | Token/sec: 1728.103
    Step:   410 | Loss: 5.4032 | lr = 0.000373 | Norm = 3.102606 | Time: 37.8225sec | Token/sec: 1732.724
    Step:   411 | Loss: 5.4521 | lr = 0.000372 | Norm = 3.604180 | Time: 37.9740sec | Token/sec: 1725.811
    Step:   412 | Loss: 5.4369 | lr = 0.000371 | Norm = 3.006497 | Time: 37.9573sec | Token/sec: 1726.570
    Step:   413 | Loss: 5.1422 | lr = 0.000370 | Norm = 6.399672 | Time: 38.0252sec | Token/sec: 1723.490
    Step:   414 | Loss: 5.3661 | lr = 0.000369 | Norm = 3.659746 | Time: 38.0295sec | Token/sec: 1723.294
    Step:   415 | Loss: 5.3258 | lr = 0.000368 | Norm = 2.976867 | Time: 37.9898sec | Token/sec: 1725.095
    Step:   416 | Loss: 5.4409 | lr = 0.000367 | Norm = 3.567602 | Time: 38.0165sec | Token/sec: 1723.884
    Step:   417 | Loss: 5.3421 | lr = 0.000366 | Norm = 3.233540 | Time: 37.9960sec | Token/sec: 1724.814
    Step:   418 | Loss: 5.4243 | lr = 0.000365 | Norm = 2.781129 | Time: 38.0283sec | Token/sec: 1723.346
    Step:   419 | Loss: 5.3577 | lr = 0.000364 | Norm = 2.511667 | Time: 37.9889sec | Token/sec: 1725.135
    Step:   420 | Loss: 5.3375 | lr = 0.000363 | Norm = 2.688940 | Time: 38.0012sec | Token/sec: 1724.579
    Step:   421 | Loss: 5.4013 | lr = 0.000362 | Norm = 3.582464 | Time: 37.9891sec | Token/sec: 1725.127
    Step:   422 | Loss: 5.3379 | lr = 0.000361 | Norm = 2.778356 | Time: 37.9953sec | Token/sec: 1724.845
    Step:   423 | Loss: 5.2828 | lr = 0.000360 | Norm = 2.588487 | Time: 38.0698sec | Token/sec: 1721.468
    Step:   424 | Loss: 5.3908 | lr = 0.000359 | Norm = 2.946087 | Time: 37.9905sec | Token/sec: 1725.064
    Step:   425 | Loss: 5.3731 | lr = 0.000359 | Norm = 2.842708 | Time: 38.0175sec | Token/sec: 1723.836
